# Same-Time Sky-to-Science Coefficient Transfer with a Physically Split Continuum

**Variant status.** This notebook is the split-zodi branch of the deployed
symmetric dual-encoder group-head regressor. The corpus has been re-decomposed
with `sky_decomp.SkyDecompLSFSurfaceIterative(split_zodi=True)`, and the ML
model learns to predict the resulting two-family continuum representation.
See the identifiability notebook
`notebooks/moon_zodi_split_identifiability.ipynb` for the decomposition-side
design and validation.

## Problem statement

Every science exposure taken with LVM at Las Campanas is accompanied by two
simultaneous sky-arm measurements — a SKY_NEAR fibre bundle pointed a few
degrees from the science IFU and a SKY_FAR bundle pointed several degrees
further away.  The two auxiliary pointings exist because almost every LVM
science target is a low-surface-brightness Galactic diffuse emitter for
which the sky is a first-order contaminant, and the goal of the sky-subtraction
pipeline is to remove that contaminant at the pixel level.

The sky is however not spatially uniform on the two-arm-to-science baseline of
five to ten degrees.  Scattered moonlight has strong angular structure driven
by the Krisciunas–Schaefer scattering phase function; the zodiacal continuum
swings by tens of percent across the ecliptic longitude range spanned by three
pointings on the same night; the OH airglow bands are modulated by mesospheric
gravity waves on scales of tens of arcminutes.  A raw copy-of-nearest-arm
sky subtraction therefore leaves systematics of order one to ten percent of
the sky flux — enough to bury the extended emission that motivates the
survey.

This notebook trains an ML transfer that takes the two sky-arm spectra
(already decomposed by the physics-driven pipeline of Chapter 1 into per-emitter
coefficient vectors) plus the row's astrometry, moon geometry, ecliptic
geometry, solar-activity indices and observing time, and predicts what the
sky spectrum would have looked like at the science IFU location at the
exposure's midpoint.  Reconstructing that predicted sky and subtracting it
pixel-by-pixel from the science exposure is what closes the sky-subtraction
loop.

---

# Chapter 1 — Decompositions

Chapter 1 turns each row's raw sky spectrum into a compact, physics-grounded
per-row coefficient vector that Chapter 3's ML transfer consumes.  The
decomposition solves one QP per row against a fixed physical dictionary of
moon, zodi, airglow and continuum templates, and persists both the
coefficients and their joint covariance.

## 1.1 Goal and pipeline

LVM's science fibres and its two sky arms (SKY_NEAR, SKY_FAR) sample three
lines of sight simultaneously. Each row (fibre × exposure) is fit
**independently** by the same physical model to yield a coefficient vector
$\mathbf{c}\in\mathbb{R}^{P}$ with $P=388$ on the deployed `new-oh`
corpus (see §1.3). The fit is a
constrained quadratic program (QP) driven by the observed flux and its
inverse-variance plus curvature regularisers. The output corpus for this variant lives at
`skysub/new-oh/` with suffix `_lsf_surface_iterative_split_zodi`
(set by `cfg.data.decomp_data_root`; earlier runs used `skysub/moon_zodi_spline/`).

The corpus we consume here is produced by

```
python skysub/decompose_parallel.py \
    lvmsframe_median_stack_1.2.1_p40_p70.fits \
    sky_decomp/data \
    --fit-model lsf-surface-iterative-split-zodi \
    --n-spline-knots 11 --n-zodi-spline-knots 1 \
    --n-refinement-cycles 5 --n-workers 32 \
    --output-dir moon_zodi_spline/
```

**Identifiability defaults (2026-09-03).** For `--fit-model
lsf-surface-iterative-split-zodi` the command above now also applies four
constraints, set in `decompose_parallel.SPLIT_ZODI_*` and needing no CLI
flags: adjacent-knot ratio bounds on both spline families
(`moon_ratio_bound = zodi_ratio_bound = 0.7`), a bracket on the moon's share
of the moon+zodi continuum (`amp_prior_tol = 3`), and an absolute bracket on
`int(zodi)` against the Leinert B500 prediction (`zodi_amp_bound = 2`).  The
last two are per-row geometry predictions installed by
`set_amplitude_prior`, computed from moon altitude/airmass, moon-target
separation, lunar phase and the target's helioecliptic coordinates via
`sky_decomp.moon_zodi_model.geometry_amplitude_prior`.

Measured on 200 lunation-stratified sky spectra, against the previous
unconstrained split:

| | before | after |
|---|---|---|
| moon/zodi colours in the WRONG order (moon-up, n=168) | 164 | **2** |
| fitted moon log-log slope (physics: −3.7) | +0.21 | **−4.10** |
| fitted zodi log-log slope (physics: −0.3) | −4.04 | **−1.64** |
| moon share with the moon >5° BELOW the horizon | 0.45 | **0.02** |
| ρ(fitted zodi, Leinert B500) | 0.09 | **0.90** |
| ρ(fitted zodi, lunar illumination) — should be ~0.1 | 0.94 | **0.12** |
| median rms ratio | 1.000 | 1.010 |

The rms cost is concentrated entirely at bright moon (×1.49 median for
FLI > 0.8, ≈1% of the continuum, blue-weighted); part of it is the old fit
overfitting by driving the moon spline to ~0 mid-band and re-using it as
piecewise scratch space, which the ratio bounds forbid.

These change the *coefficients*, not the design matrix: no `set_target_airmass`
is applied, so the basis is identical for every row and every reconstruction
path (`data.py`, `trainer.py`, `wavelengths.py`, the diagnostics cells) is
unaffected.  **The corpus must be regenerated and the ML retrained** — both
the moon/zodi split and the moon+zodi sum move.

## 1.2 Physical model

The decomposition treats the sky as a non-negative linear combination of a
handful of physical carriers whose spectral shapes are fixed by physics or
by pre-fit lookup tables; only the per-row amplitudes are free parameters.

The decomposition writes each row's spectrum $y(\lambda)$ as a non-negative
sum of physical templates:

$$
y(\lambda) \;\approx\; \sum_k c_k\; T_k(\lambda),\qquad c_k\ge 0,
$$

where $T_k$ is the LSF-convolved template of family $k$. All families share
the row's Gaussian LSF sigma $\sigma_{\rm lsf}(\lambda)$ (see §1.6).

### 1.2.1 Moon carrier (15 basis functions)

Scattered moonlight is the dominant broadband contaminant on bright-moon
nights.  Its spectrum is the solar SED reddened through the lunar-albedo
envelope; the free parameters are the 15 B-spline amplitudes that let the
overall shape drift smoothly with the row's specific scattering geometry.

$$
T_{\rm moon,\,j}(\lambda) \;=\; I_\odot(\lambda)\; \alpha_{\rm ROLO}(\lambda,\,\phi_0{=}30°)\; B_j^{(K_{\rm m}=11)}(\lambda),
$$

with $I_\odot$ the solar SED (Meftah 2018 UV–NIR reference), $\alpha_{\rm ROLO}$
the disk-integrated lunar albedo at a fiducial phase angle, and
$B_j^{(K_{\rm m})}$ a cubic B-spline basis with 11 interior knots (15 columns) after
the spline3 knot-count reduction (2026-08-25; the pre-reduction default was
25 knots / 29 columns). Flux-fit quality on bright-moon and moon-down regimes
stays within 1–3% of the pre-reduction baseline while eliminating all
adjacent-knot |corr| > 0.99 pairs. The ROLO envelope carries sharp mineral
absorption bands, giving the moon carrier a distinctive spectral shape.

### 1.2.2 Zodi carrier (5 basis functions)

The zodiacal light is a smooth reddened solar continuum; separating it from
the moon carrier lets the ML head learn its ecliptic-geometry dependence
independently from moon geometry.  Pre-split, the smooth zodi component was
silently absorbed into `Moon_bs` on moon-down + low-$|\beta_{\rm ecl}|$ rows.

$$
T_{\rm zodi,\,j}(\lambda) \;=\; I_\odot(\lambda)\; (\lambda/5000\,\text{\AA})^{0.26}\; B_j^{(K_{\rm z}=1)}(\lambda),
$$

with $K_{\rm z}=1$ interior knot (5 columns) after the spline3 reduction
(2026-08-25; the pre-reduction default was 3 knots / 7 columns) and $0.26$
the Leinert-style zodiacal reddening exponent. The 1-knot spline is heavily
curvature-penalised ($\lambda_{\rm z}=0.1$, §1.4) so the zodi carrier is
nearly monotone in $\lambda$. Introducing this family (`split_zodi=True`)
is the central decomposition-side change of this variant: pre-split, the
smooth zodi
component was silently absorbed into `Moon_bs` on moon-down + low-$|\beta_{\rm ecl}|$
rows, giving `c_moon` a mixed identity that the ML head could not learn.

### 1.2.3 OH Meinel bands

OH nightglow is the dominant narrow-line airglow signal at all wavelengths
beyond ~6000 Å.  LVM's resolution resolves individual rotational
transitions, and each carries its own free amplitude so that the fit can track
mesospheric temperature and gravity-wave-driven line-ratio changes.

Rotationally-grouped OH transitions from the PMD `pmd_popmodel_OH` file
produce ~110 stick spectra $L_i(\lambda)$; these are convolved with the row
LSF and stacked into `matrix_oh`. Coefficients are `OH_ddd` (rotational
level).

### 1.2.4 Diffuse continuum templates

Three fixed diffuse templates $D_1,D_2,D_3$ (HO2, FeO, O2Ac) come from
`pmd_refcont`. They carry the flat, wavelength-smooth part of the continuum
so `Moon_bs` and `Zodi_bs` compete only for curvature-sensitive shape.

### 1.2.5 Molecular O2 and atomic lines

Two O2 stick libraries ($O_2$, $O_2^b$) and two atomic families (K I 7699,
N I 5199) round out the design matrix. Names: `O2_bd`, `ATOM_K`, `ATOM_N`.

## 1.3 Coefficient block and design matrix

Concatenating the templates gives the row-level design matrix
$\mathbf{A}\in\mathbb{R}^{N_\lambda\times P}$:

$$
\mathbf{A} \;=\; [\, \mathbf{A}_{\rm oh}\ |\ \mathbf{A}_{\rm moon}\ |\ \mathbf{A}_{\rm zodi}\ |\ \mathbf{A}_{\rm diff}\ |\ \mathbf{A}_{\rm atom}\ |\ \mathbf{A}_{\rm orc}\ |\ \mathbf{A}_{\rm o2}\,].
$$

For this variant $N_\lambda=12401$ pixels on 3600–9800 Å at $\Delta\lambda=0.5$ Å,
and $P=388$ coefficients on the deployed `new-oh` corpus (spline3 reduction,
2026-08-25; the OH block is 358 columns after the re-decomposition, against
~403 on the older `moon_zodi_spline/` corpus where $P=433$):

| block         | count | code family    | physical driver                           |
|:--------------|------:|:---------------|:------------------------------------------|
| OH Meinel     |   ~400| `OH_ddd`        | thermospheric temperature, geomagnetic $A_p$ |
| Moon spline   |    15 | `Moon_bs\d+`    | $g_{\rm moon}(\phi, h, r_{\rm sep})$      |
| Zodi spline   |     5 | `Zodi_bs\d+`    | $B_{500}(\lambda_\odot^{\rm rel}, |\beta_{\rm ecl}|)$·airmass |
| Diffuse       |     3 | HO2, FeO, O2Ac  | mesospheric chemistry residual            |
| Atomic K I    |     3 | `ATOM_K`        | K I 7699 nightglow                        |
| ORC / N I     |     4 | `ATOM_N`        | N I 5199 nightglow                        |
| Molecular O2  |     3 | `O2_bd`         | O2 Meinel                                 |

Deployed knot counts: `n_spline_knots=11` for moon and
`n_zodi_spline_knots=1` for zodi, auto-inferred from `coef_names` by the
`infer-spline-knots` notebook cell.

## 1.4 QP formulation

For each row, the fit solves a **weighted non-negative regularised
least-squares** problem:

$$
\boxed{\;\;\mathbf{c}^\star \;=\; \arg\min_{\mathbf{c}\ge 0}\;\; \underbrace{\tfrac{1}{2}\big\lVert \operatorname{diag}(\sqrt{\mathbf{w}})\big(\mathbf{A}\mathbf{c}-\mathbf{y}\big)\big\rVert_2^2}_{\text{data likelihood}} \;+\; \underbrace{\lambda_{\rm m}\lVert \mathbf{D}^{(2)}_{\rm m}\mathbf{c}_{\rm m}\rVert_2^2 \;+\;\lambda_{\rm z}\lVert \mathbf{D}^{(2)}_{\rm z}\mathbf{c}_{\rm z}\rVert_2^2}_{\text{spline curvature}}.\;\;}
$$

Symbols:

- $\mathbf{y}\in\mathbb{R}^{N_\lambda}$: the observed spectrum, scaled by
  `FACTOR = 1e14` (to bring counts to O(1)).
- $\mathbf{w} = 1/\sigma_{\rm pix}^2$: pixel inverse-variance from the
  reduction pipeline (`IVAR`, `SKY_IVAR`), with a relative floor
  `PIXEL_SIGMA_FLOOR_REL` per HDU.
- $\mathbf{D}^{(2)}$: second-difference operator on adjacent spline
  coefficients; suppresses ripples/oscillations in the moon and zodi splines.
- $\mathbf{c}\ge 0$: hard non-negativity constraint on every column
  (every family is physically non-negative).

The default curvature strengths (per unit scaled flux $y'=y/d$, see §1.5) are

$$
\lambda_{\rm m}=10^{-3},\qquad \lambda_{\rm z}=10\,\lambda_{\rm m}=10^{-1}.
$$

The heavier zodi penalty reflects that with only $K_z=3$ interior knots the
zodi spline should encode a smooth departure from the power-law, not sharp
features.

### 1.4.1 Solver

The problem is convex and is delegated to **Clarabel** (interior-point QP,
sparse triangular Hessian). The active-set structure at the optimum is used
downstream to compute coefficient uncertainties (§1.9).

### 1.4.2 Interline reweighting

The fit optionally reweights pixels near strong OH lines by
`moon_interline_weight ∈ [0, 1]` to prevent line residuals from stealing
signal from `Moon_bs`. This is disabled in the deployed split-zodi
configuration but the option remains in `fit.py`.

### 1.4.3 Identifiability constraints on the moon/zodi split

The two continuum families are nearly degenerate. Both are built on the same
solar template, so the fit can trade flux between them almost freely, and the
data term alone does not pin the split: the *sum* is well constrained while the
*division* is not. Measured on 200 lunation-stratified sky spectra, the
unconstrained fit put the two colours in the **wrong order on 164 of 168
moon-up spectra** — fitted moon log-log slope $+0.21$ where the physics gives
$-3.7$, fitted zodi $-4.04$ where the physics gives $-0.3$. It also gave the
moon family 45% of the continuum with the moon 37° *below* the horizon, and
produced a "zodi" whose amplitude tracked lunar illumination at
$\rho=+0.95$ while retaining only $\rho=0.09$ of its Leinert $B_{500}$
dependence. The downstream ML was therefore being handed moon-labelled zodi
spectra and vice versa.

Three families of constraint fix this. All are **hard linear inequalities**
appended to the same non-negative cone as $\mathbf{c}\ge 0$, not penalties:
they never bid against the likelihood, and the first two are homogeneous in
$\mathbf{c}$ and so carry no dependence on the flux scale.

**(a) Adjacent-knot ratio bounds — fix the colours.** For each family
$f\in\{\rm m, z\}$ and each adjacent pair of spline coefficients,

$$
\beta_f\,c^{(f)}_k \;\le\; c^{(f)}_{k+1} \;\le\; \beta_f^{-1} c^{(f)}_k ,
\qquad \beta_{\rm m}=\beta_{\rm z}=0.7 .
$$

This caps how fast a family's multiplier may vary with wavelength, so neither
family can manufacture the other's colour. With $K$ basis functions the total
permitted swing across the band is $\beta^{-(K-1)}$, so the *same* $\beta$ is
looser the more knots there are — these values are calibrated for 15 moon and
5 zodi basis functions and must be re-validated if the knot counts change.
The bounds also forbid a spline going to zero mid-band and re-emerging, which
is how the unconstrained fit reached a lower residual: it used the moon family
as piecewise scratch space, collapsing it to $\sim 0$ across 5100–6900 Å.
Note the interaction with non-negativity: since $c\ge0$ already holds, a ratio
bound makes each block either all-positive or identically zero.

**(b) Moon-share bracket — fixes dark time.** With
$u=\int\!{\rm moon}\,{\rm d}\lambda$ and $v=\int\!{\rm zodi}\,{\rm d}\lambda$,
both linear in $\mathbf{c}$,

$$
f_{\rm lo} \;\le\; \frac{u}{u+v} \;\le\; f_{\rm hi},
\qquad\Longleftrightarrow\qquad
\begin{cases}
(1-f_{\rm hi})\,u - f_{\rm hi}\,v \le 0\\[2pt]
f_{\rm lo}\,v - (1-f_{\rm lo})\,u \le 0
\end{cases}
$$

centred on the geometry-predicted fraction $f_{\rm pred}$ with a
multiplicative tolerance $\kappa_f=3$, clipped into
$[\epsilon,\,1-\epsilon]$ with $\epsilon=0.02$. Constraining the *fraction*
rather than either amplitude is what makes this calibration-free: both
predictions leave the physical model through the same solid-angle and flux
conversion, so the ratio is independent of throughput. The $\epsilon$ floor
guards the two places the prediction is least trustworthy — the below-horizon
suppression is a learned extrapolation fitted only to $-7°$ altitude, and ROLO
phase coverage runs out on thin crescents.

**(c) Absolute Leinert anchor — fixes the amplitude geometry.**

$$
\kappa_z^{-1} Z_{\rm pred} \;\le\; v \;\le\; \kappa_z\,Z_{\rm pred},
\qquad \kappa_z = 2 .
$$

The bracket in (b) is *relative*, so it can only divide whatever total the two
splines hold; it cannot stop both growing with the moon. Only an absolute
anchor reaches that. Its empirical licence is dark time: with the shape bounds
on and no moon, fitted/predicted $\int\!{\rm zodi}$ is $0.94$ with $0.16$ dex
scatter, so the literature scale is good to ~6% median with a factor-1.4
spread — do not tighten $\kappa_z$ below that spread. These are the only rows
with a non-zero right-hand side, hence stated in native flux units:
`set_amplitude_prior` must be given $Z_{\rm pred}$ on the same scale as the
flux being fitted.

Both predictions come from a single call to the frozen physical model with its
*learned* scale factors set to unity, keeping only the below-horizon taper.
The other ten parameters were fitted with `CORRECTION_SCOPE='moon_plus_zodi'`,
i.e. only the sum was ever constrained, so the split they imply is not
independently calibrated; and `zodi_target_airmass_log = 1.28` (zodi growing
as $X^{1.28}$, where extinction should make it *fall*) looks like it absorbed
the very moon-into-zodi leakage these priors exist to remove.

**Measured effect** (200 spectra, 168 moon-up, 28 dark):

| quantity | before | after | target |
|---|---|---|---|
| colour reversals, moon-up | 164/168 | **2/168** | 0 |
| separation margin $s_z-s_{\rm m}$, p10 | $-7.71$ | $+0.98$ | $>0$ |
| moon log-log slope | $+0.21$ | $-4.10$ | $-3.7$ |
| zodi log-log slope | $-4.04$ | $-1.64$ | $-0.3$ |
| moon share, moon $>5°$ below horizon | 0.45 | **0.02** | $\sim0$ |
| $\rho(\int\!{\rm zodi},\,B_{500})$ | 0.09 | **0.90** | $\sim1$ |
| $\rho(\int\!{\rm zodi},\,{\rm FLI})$ | 0.94 | **0.12** | 0.10 |
| near–far moon disagreement, $p_{50}$ | 0.0554 | **0.0045** | — |

The cost is a median residual ratio of 1.01, concentrated entirely at bright
moon (×1.49 median for FLI > 0.8, ≈1% of the continuum, blue-weighted). Part
of that is the old fit overfitting through the spline hole described in (a),
so the comparison flatters the unconstrained version.

**Not adopted, and why.** A colour envelope carrying the aerosol scattering
channel and a multiple-scattering enhancement was implemented and measured
neutral: 15 moon knots at $\beta=0.7$ can already manufacture the
separation-dependent colour themselves, so the envelope was never the binding
constraint. A blue-window relaxation of the moon bound (optionally gated on
moon dominance) buys ~1.5%, because the moon spline never reaches its blue
bound — the *amplitude* anchor is what binds there. Both remain in `fit.py`,
off by default.

**Residual weaknesses.** The 2/168 remaining reversals are near-ties, not
confident inversions: over the full 17k-row corpus the worst separation is
$-1.68$ against $-9.95$ before, and 100% lie within 2.0 of the boundary. Their
rate rises with decreasing lunar illumination (0.1–0.7% at FLI > 0.85, 9–15%
below 0.3) and with decreasing moon altitude (2.2% above 40°, **19.4% below
5°**), and the science arm reverses 3.6× more often than the far sky arm
(4.7% vs 1.3%) — consistent with faint-source contamination of the science
continuum. Rows reversed in any arm are dropped from training and from the
every10 diagnostic samples (§3.8).

## 1.5 Linear algebra: rescaling for stability

Direct QP on physical units suffers from column-scale imbalance
(order-of-magnitude differences between OH sticks and B-spline columns).
`SkyDecomp._solve_nonnegative_weighted` applies a **two-stage rescaling**:

1. **Data scale.** Let $d = \max\big(\sqrt{\overline{y_w^2}},\,1\big)$
   where $y_w = \sqrt{\mathbf{w}}\odot\mathbf{y}$. Set $\mathbf{y}' = \mathbf{y}/d$.
2. **Column scale.** Let $s_k = \lVert (\sqrt{\mathbf{w}}\odot\mathbf{A})_k\rVert_2$
   per column. Set $\mathbf{A}'_{:,k} = \mathbf{A}_{:,k}/(d\,s_k)$.
3. Solve $\mathbf{c}'^\star = \arg\min_{\mathbf{c}'\ge 0} \tfrac{1}{2}\lVert\sqrt{\mathbf{w}}(\mathbf{A}'\mathbf{c}'-\mathbf{y}')\rVert^2 + \text{(rescaled regularisers)}$
   using Clarabel.
4. **Undo the rescaling:** $c_k = c'_k / s_k$.

The regularisation operators are rescaled consistently:

- Curvature: $\mathbf{D}^{(2)}_{\rm scl} = \mathbf{D}^{(2)}\,\operatorname{diag}(1/s_{\rm m})$
  (and similarly for zodi).

## 1.6 LSF-surface-iterative refinement loop

The row's line-spread-function width varies with wavelength and with fibre
position on the focal plane, and it sets the shape of every narrow template
the QP solves against.  Because the LSF and the coefficients constrain each
other, the pipeline updates them alternately until both stabilise.

`SkyDecompLSFSurfaceIterative` alternates between:

1. **QP fit** of $\mathbf{c}^\star$ given the current LSF $\sigma_{\rm lsf}(\lambda)$
   and continuum baseline.
2. **LSF-surface update.** Extract a low-order surface in
   $(\lambda,\text{fibre})$ from the OH-line residual widths; refit
   $\sigma_{\rm lsf}$ per row.
3. **Continuum baseline update.** Small local update of the diffuse family
   from the low-pass residual.

The loop runs `n_refinement_cycles = 5` by default. Each cycle refits the
same design matrix $\mathbf{A}$ (rebuilt with the updated LSF); the QP is
warm-started from the previous solution. Convergence is monitored by the
relative change in `chi2_reduced`.

## 1.7 Weights

The QP weights every pixel by its inverse variance so that noisy pixels
contribute little to the fit while well-measured ones drive it.  The weight
schedule below is straightforward, but its outputs feed the coefficient
covariance in §1.9 and the ML per-element weights in §3.6.3, so we spell it
out.

The data-side weights carry through the pipeline as inverse-variance:

- **Base ivar**: from `IVAR` / `SKY_IVAR` HDUs, floored by
  `PIXEL_SIGMA_FLOOR_REL · (typical pixel signal)` per HDU family
  (`PIXEL_SIGMA_HDU_NAMES`).
- **Interline boost** (optional): $w_i \to w_i\cdot m^{\rm il}(\lambda_i)$
  with $m^{\rm il}<1$ on strong OH cores; disabled here.
- **Finite mask**: only pixels with $y_i,\sigma_i,w_i$ all finite and
  $w_i>0$ enter the QP.

## 1.9 Coefficient uncertainties

Every ML weight and diagnostic downstream needs to know how well the per-row
decomposition constrained each coefficient.  §1.9 defines the per-row
covariance we persist to disk and the way it degrades gracefully when a
coefficient sits at its non-negativity boundary.

At the QP optimum, split coefficients into active (interior, $c_k>0$) and
inactive (pinned at 0, $c_k=0$). On the active subset $\mathcal{A}$, the
regularised Fisher information is

$$
\mathbf{F}_{\mathcal{A}} \;=\; \mathbf{A}_{:,\mathcal{A}}^{\top}\operatorname{diag}(\mathbf{w})\mathbf{A}_{:,\mathcal{A}} \;+\; 2\lambda_{\rm m}\mathbf{D}^{(2)\top}\!\mathbf{D}^{(2)}\big|_{\mathcal{A}} \;+\; 2\lambda_{\rm z}\mathbf{D}^{(2)\top}\!\mathbf{D}^{(2)}\big|_{\mathcal{A}} \;+\; \ldots
$$

The covariance is $\mathbf{F}_{\mathcal{A}}^{-1}$, inflated by
$\max(\chi^2_\nu,\,1)$ to absorb model misfit. Column-scale and data-scale
are undone before reporting; inactive coefficients receive `NaN`. See
`SkyDecomp._coef_err_active_set`. These uncertainties feed the ML loss
weights via `COEF_ERR` (§3.6.2).

**Joint covariance persistence.** The diagonal $\boldsymbol\sigma_{\rm coef}=\sqrt{\operatorname{diag}(\mathbf{F}_{\mathcal{A}}^{-1})}$ discards all off-diagonal information; for a near-collinear basis (adjacent Moon_bs
knots stay at $|\rho|\!\sim\!0.88$ even at the reduced $K\!=\!11$), that
marginal $\sigma$ substantially over-states the constraint on any
individual knot while under-stating the constraint on any joint quantity
(block amplitude, PCA scores, LSF-convolved flux over the moon block).
`SkyDecomp._coef_err_active_set` therefore also materialises, on the same
solve-group Hessians, the FULL active-set covariance restricted to the
moon and (when `split_zodi=True`) zodi B-spline blocks:

$$
\mathbf{\Sigma}_g \;=\; \big[\mathbf{F}_{\mathcal{A}}^{-1}\big]_{\mathcal{A}\cap g,\;\mathcal{A}\cap g}\qquad g\in\{\text{moon},\text{zodi}\}
$$

with the same column-scale and per-column $\chi^2$ inflation applied
consistently on the diagonal and off-diagonal (so
$\sqrt{\operatorname{diag}(\mathbf{\Sigma}_g)}$ reproduces the entries
of $\boldsymbol\sigma_{\rm coef}$ in that block); inactive rows/columns
are `NaN`.  These matrices are persisted per row to disk as the
`COEF_COV_MOON` and `COEF_COV_ZODI` ImageHDUs by `results_to_fits`
(shape $n_{\rm row}\!\times\!n_g\!\times\!n_g$; skipped for the
degenerate $n_g\!=\!1$ physical moon-zodi model).  Per-row storage
overhead at deployed dimensions is 15×15+5×5 = 250 float64 per row
$\approx 2\,\text{kB}$; a full corpus adds $\sim$100 MB across the
three arms.  The compact `*_meta_coef_*.fits` files carry both HDUs
through `extract_meta_and_coef_products`.  See §3.6.2 for how the
notebook consumes them.


## 1.10 Input data

Each row's inputs to the fit are:

- $\mathbf{y}$: 12401-pixel median-stacked flux on 3600–9800 Å.
- $\mathbf{w}$: per-pixel inverse variance.
- $\sigma_{\rm lsf}(\lambda)$: LSF Gaussian sigma per pixel (updated
  iteratively).
- Row metadata: MJD (for solar-activity, ecliptic geometry), fibre RA/Dec
  (for airmass, moon separation, ecliptic latitude), exposure duration.

The corpus consumed here is
`skysub/moon_zodi_spline/lvmsframe_median_stack_1.2.1_p40_p70*.fits`, all
with suffix `_lsf_surface_iterative_split_zodi` and containing a new
`COMP_ZODI` HDU. The wavelength cache is
`moon_zodi_spline/coef_wavelengths_basis_v4.npz`.

---

## 1.11 Row filtering

`data.apply_triplet_filters` gates the training corpus. Of 17 214 rows, 10 249
survive. The gates, in the order they are reported:

- **Field exclusion** — 10° around the LMC and SMC (−22.8%): both combine
  bright stellar populations, dense H II regions and diffuse ionised gas at
  velocities close enough to airglow to blend with the science-arm fit.
- **Physical context sanity** — altitude $\ge 0$, airmass $\le 3$, van Rhijn
  factors finite and $\ge 1$, in all three arms (−0.1%).
- **Reduced $\chi^2$** — max across arms, capped at the 90th percentile
  (−10%). Any arm's decomposition failure disqualifies the observation.
- **Moon/zodi role reversal** (−2.5%, 434 rows) — see below.
- **Hard coefficient clips** on `feo` and `atom_k`, and an OH per-row-mean
  MAD filter (κ=4) that removes rows whose OH block ran away by factors of
  $10^6$–$10^{15}$.
- **κσ clipping** (κ=6, 3 iterations) on the concatenated three-arm
  coefficient vector.

### 1.11.1 The moon/zodi reversal gate

A row is *reversed* when its fitted moon continuum is redder than its fitted
zodi continuum, $s_{\rm m} > s_{\rm z}$, where $s$ is the log-log slope of the
reconstructed component. The physics puts the moon near $-3.7$ and the zodi
near $-0.3$, so the ordering is unambiguous whenever both families carry flux.

Such a row is not merely noisy, it is **mislabelled**: its moon coefficients
describe zodiacal light. Training on it teaches the network a moon-geometry
mapping from zodi-shaped targets, and scoring a prediction against it measures
nothing. The gate therefore belongs with the $\chi^2$
decomposition-failure gate rather than with the target-outlier filters, and it
is applied identically to the every10 diagnostic samples (§3.8) — otherwise
those samples surface spurious outliers the model never trained on.

The test only applies where both families carry flux, $0.05 \le u/(u+v) \le
0.95$; a slope is meaningless when a component is switched off, and under the
decomposition priors (§1.4.3) dark-time rows sit at a moon share of ~0.02 by
construction. Rows failing that condition are kept and counted separately.
Reversal in **any** of the three arms disqualifies the row, matching the
$\chi^2$ convention.

On the deployed corpus: 434 of 17 214 rows dropped (2.5%), which is 5.5% of
the 7 831 rows where the test applies. Per arm the rate is 2.2% (near), 1.3%
(far) and 4.7% (science), and 70% of flagged rows reverse in only one arm —
so the per-arm rate is the honest one and the residual reversals are
per-fit noise rather than a geometry-driven failure.


# Chapter 2 — Geometric normalisation & effective extinction

**Framing principle.** The ML transfer (Chapter 3) is trained and evaluated
in **intrinsic-emissivity space**, not in observed-flux space. Each row's
decomposition coefficients (Chapter 1) are pre-divided by the row's own
line-of-sight geometry factor $V(z;h)\cdot 10^{-0.4 k(X-1)}$ **before** they
enter the network, so the network sees the zenith-equivalent amplitude of the
underlying emitting layer. On the output side, predictions are multiplied
back by the science-arm geometry factor to restore the **observed** amplitude
at the science pointing:

$$
\boxed{\;\;\mathbf{c}^{\rm intrinsic}_{r,g} \;=\; \frac{\mathbf{c}^{\rm obs}_{r,g}}{V(z_r;\,h_g)\cdot 10^{-0.4\,k_g\,(X_r-1)}} \;\;\xrightarrow{\;\text{ML}\;}\;\; \hat{\mathbf{c}}^{\rm intrinsic}_{{\rm sci},g} \;\;\xrightarrow{\;\times V(z_{\rm sci};h_g)\cdot 10^{-0.4\,k_g\,(X_{\rm sci}-1)}\;}\;\; \hat{\mathbf{c}}^{\rm obs}_{{\rm sci},g}.\;\;}
$$

The reason this framing works is that intrinsic emissivity is the quantity
that **actually transfers** from one line of sight to another: gravity-wave
fluctuations aside, the emitting layer at 87 km looks the same in whatever
direction you point during a 900-s exposure. Zenith angle, altitude and
extinction on the other hand differ between the two sky arms and the science
arm, and if the network had to learn those differences from context it would
be re-deriving well-known atmospheric physics on limited training data. By
dividing them out **in closed form before the network**, we let the ML head
focus on the remaining physics — gravity-wave and short-timescale variability
that the sky arms can genuinely constrain for the science arm.

The decomposition (Chapter 1) returns coefficients in **observed physical
units**: amplitudes at the actual pointing, at the actual zenith angle, at the
actual airmass. Chapter 2 is the closed-form transform that converts those to
the intrinsic emissivities the ML consumes, and back again on the output side.

## 2.1 The scaling law

The intrinsic-emissivity transform of §2.2 rests on the standard thin-shell
airglow scaling law: line-of-sight-integrated shell emission at zenith angle
$z$ enhances by a $\sec$-like factor $V(z;h)$ that saturates near the horizon
because the observer sits inside a curved shell rather than an infinite plane.

For an airglow layer at zenith angle $z$ and target airmass $X(z)$, the
observed amplitude relates to the intrinsic (zenith-equivalent) emissivity by

$$
A_{\rm obs} \;=\; A_{\rm int}\;\underbrace{V(z;h)}_{\text{slant path}}\;\underbrace{10^{-0.4\,k(\lambda)\,[X(z)-1]}}_{\text{extinction}},
$$

with the **van Rhijn** slant-path factor for a thin shell at height $h$ above
Earth radius $R_\oplus$

$$
V(z;h)\;=\;\left[1-\left(\frac{R_\oplus}{R_\oplus+h}\right)^{\!2}\sin^2 z\right]^{-1/2},\qquad V(0;h)=1.
$$

$V$ **saturates** toward the horizon (6.0 at $h=87$ km, 3.4 at $h=285$ km)
rather than diverging like $\sec z$, because the observer sits inside a curved
shell. Using plane-parallel airmass instead overstates the enhancement by 4%
at $z=60°$ for a mesospheric layer and 12% for an ionospheric one — a smooth
function of $z$, so it does not average away and would alias into whatever the
network learns as geometry.

## 2.2 Where it is applied

The training pipeline consumes coefficients from three simultaneous pointings
taken at different zenith angles.  If the network had to learn the geometry
trigonometry from the context vector we would be re-deriving well-known
physics on limited data; §2.2 spells out where the closed-form correction is
inserted so it stays consistent with the compressor and RobustScaler that
sit on either side of it.

`airglow_geometry_scale()` returns $V\!\cdot\!10^{-0.4k(X-1)}$ per row and per
coefficient, and is applied **in physical units, before the square-root /
asinh transforms and before robust scaling**. Neither of those operations
commutes with a division by the geometry factor: the scaler subtracts a
median, so $(\sqrt{c}-m)/V\ne\sqrt{c/V}-m$, and under the square root the
correct divisor would be $\sqrt{V}$ rather than $V$. Applying the correction
downstream of either transform produces an error of order $V$ itself.

The full training-time pipeline for each airglow group therefore is:

1. **Decompose** each row → observed coefficients $\mathbf{c}^{\rm obs}_{r,g}$.
2. **Divide out geometry** (this chapter):
   $\mathbf{c}^{\rm int}_{r,g} \;=\; \mathbf{c}^{\rm obs}_{r,g}\,/\,\big[V(z_r;h_g)\cdot 10^{-0.4k_g(X_r-1)}\big]$
   for both sky arms and the science arm, independently.
3. **Compress** (§3.3): elementwise transform + PCA truncation.
4. **Robust-scale** across the training set.
5. **ML forward** (§3.4): predict $\hat{\mathbf{s}}^{\rm int}_{\rm sci,g}$ (still in intrinsic-emissivity, robust-scaled, compressed space).

At inference the pipeline is inverted:

6. **Inverse robust scaling**, **inverse compressor**, then
7. **Multiply back by science-arm geometry**:
   $\hat{\mathbf{c}}^{\rm obs}_{{\rm sci},g} \;=\; \hat{\mathbf{c}}^{\rm int}_{{\rm sci},g}\cdot\big[V(z_{\rm sci};h_g)\cdot 10^{-0.4k_g(X_{\rm sci}-1)}\big]$,
   inside `predict_sci_coefficients_default()`.
8. **Reconstruct** the observed sci spectrum $\hat y_{\rm sci}(\lambda) = A\hat{\mathbf{c}}_{\rm sci}$.

**The ML target is therefore the intrinsic (zenith-equivalent) emissivity of
each layer, not the observed coefficient**. The **loss** is computed on
intrinsic-emissivity residuals, not on observed residuals. Rows at high
airmass therefore contribute to the loss with the same weight as rows at
zenith, which is the physically correct behaviour: the underlying physics we
are trying to interpolate does not know the target's zenith angle. The three
`vanrhijn_*` context columns are still fed to every encoder — they are
redundant for the airglow groups (whose geometry is pre-divided) but the
`moon` and `zodi` groups need them.

Because the reference point is $X=1$ rather than $X=0$, the recovered
"emissivity" is a **zenith-equivalent** amplitude, still carrying one airmass
of extinction. This is self-consistent and cancels in the sky-to-sci transfer,
but it is not a physical layer emissivity and should not be interpreted as
one.

`assert_context_is_physical()` guards every call. It checks physical ranges
and cross-checks any `vanrhijn_*` column against van Rhijn recomputed from
`alt`. Those two are computed independently upstream, so disagreement means
the context was transformed on the way in — the failure mode being guarded
against is evaluating geometry on scaler output, where an altitude of $\sim
0.5$ becomes a zenith angle of $\sim 89.5°$ and every row silently receives a
near-horizon factor of $\sim 6$.

## 2.3 Layer heights

Each coefficient group is tagged with an emitting-layer height $h$ that
determines the van Rhijn factor its columns receive.  The values below
follow the standard airglow-modelling literature.

Assigned per group via `GROUP_HEIGHT_FEATURE`:

| group          | van Rhijn feature | height       | physics                                         |
|:---------------|:------------------|-------------:|:------------------------------------------------|
| `mesospheric`  | `vanrhijn_87km`   | 87 km        | OH Meinel + O$_2$ atmospheric bands             |
| `atomic`       | `vanrhijn_95km`   | 95 km        | K, Na, mesopause metals / metastables           |
| `ionospheric`  | `vanrhijn_285km`  | 285 km       | O I 6300/6364 and F-region recombination lines  |
| `continuum`    | `vanrhijn_87km`   | 87 km        | HO$_2$ / FeO / O$_2$Ac mesopause chemiluminescence |
| `moon`         | (none)            | factor 1.0   | scattered moonlight — scattering geometry, not thin-shell emission |
| `zodi`         | (none)            | factor 1.0   | line-of-sight integrated interplanetary dust — not a thin shell     |

The `continuum` group name is retained from an earlier grouping in which
HO$_2$ / FeO / O$_2$Ac were treated as aerosol continuum; they are actually
airglow, but they are kept in a separate coefficient group from `mesospheric`
because they use a broadband basis rather than the OH line-group basis and
their compression pipeline stays sqrt-identity (§3.3). Moon and zodi
explicitly bypass van Rhijn: neither is thin-shell emission, and their
sky-to-sci transfer relies on other physics (Krisciunas–Schaefer geometry for
moon, Leinert lookup + airmass for zodi).

## 2.4 Per-coefficient effective wavelength

Extinction varies strongly across the LVM range, so a single wavelength per
group is not adequate: the `mesospheric` group alone spans O I 5577, Na D and
the OH / O$_2$ bands, and even within the OH Meinel system $k(\lambda)$ varies
by nearly a factor of two between the reddest and bluest bands.
`resolve_coef_wavelengths_a()` assigns a wavelength per coefficient, in this
order of precedence:

1. **basis centroid** — reconstructing with $\mathbf{c}=\mathbf{e}_j$ isolates
   basis component $j$, whose intensity-weighted centroid is
   $\lambda_{\rm eff}(j)=\sum\lambda|f_j|/\sum|f_j|$. This reads the actual
   basis, which is built from the same PMD population model, LSF and
   wavelength grid the decomposition uses at fit time, so the mesopause
   rotational Boltzmann factor for OH, the exact vibrational populations, and
   the blend structure across overlapping bands are all baked in by
   construction — it is the definitionally-correct weighting. Cached to
   `WAVELENGTH_CACHE` (`coef_wavelengths_basis_v4.npz` for the split-zodi
   basis).
2. **explicit wavelength token or species lookup** baked into `COEF_SCHEMA`
   for named species (K I 7699, N I 5199, Na D, O I 5577, O I 6300, O I 7774,
   O I 8446, O$_2$ b-band). Used only for the handful of coefficients whose
   basis support is empty on the current wavelength grid.
3. **group default** — `GROUP_EFFECTIVE_WAVELENGTH_A`, last resort.

Provenance is recorded per coefficient and printed for the airglow groups,
because a silently wrong wavelength becomes a silently wrong extinction
correction.

## 2.5 Effective extinction

The extinction coefficient $k(\lambda)$ enters as $10^{-0.4k(X-1)}$. Three
physical facts govern how it should be chosen, and the implementation follows
from them.

### 2.5.1 Only the ratio matters, so the absolute value barely does

The sky-to-sci transfer of an airglow amplitude is

$$
R\;=\;\frac{V(z_{\rm sci})}{V(z_{\rm sky})}\;10^{-0.4\,k\,(X_{\rm sci}-X_{\rm sky})},\qquad \frac{\partial\ln R}{\partial k}\;=\;-0.4\ln 10\;\Delta X\;\approx\;-0.921\,\Delta X.
$$

The absolute $k$ cancels; only $k$ times the airmass **difference** survives.
The three pointings are simultaneous and share one atmosphere, so a per-night
error $\delta k$ propagates as $0.921\,\delta k\,\Delta X$. With realistic
aerosol variability ($\delta k\sim 0.02$–$0.04$ mag/airmass) and $\Delta
X\sim 0.2$ that is a 0.2–0.7% transfer error, an order of magnitude below the
gravity-wave floor. **Per-observation extinction is therefore not modelled,
and does not need to be.**

### 2.5.2 A stellar curve is the wrong curve for an extended source

For a star, scattered photons are lost. Airglow is a quasi-uniform source
filling the sky, so photons scattered out of the beam are largely replaced by
photons scattered in from adjacent lines of sight, and the effective
attenuation is well below the single-scattering value. Scattering dominates
the total everywhere airglow matters — Rayleigh + aerosol is roughly 70–100%
of $k$ from 4000 Å to 1 µm at LCO's 2380 m — so a stellar curve
over-corrects, most severely in the blue. This is a **coherent bias**, not
random scatter.

### 2.5.3 $h$ and $k$ are nearly degenerate

Over the zenith range LVM observes, $\ln V(z;h)$ and $X-1$ are collinear to
$\rho>0.995$ for $z\le 60°$. A height error can masquerade as an extinction
error and vice versa. What the data constrain, and all the ML needs, is the
product $V\cdot 10^{-0.4 k(X-1)}$.

### 2.5.4 Implementation: fit $k_{\rm eff}$ from the airglow itself

Rather than model $k$, `fit_effective_extinction()` performs a **Bouguer fit
that uses airglow as its own source**. For coefficient $j$ and a simultaneous
sky pair,

$$
\ln\!\frac{A_{\rm near}}{A_{\rm far}}\;-\;\ln\!\frac{V_{\rm near}}{V_{\rm far}}\;=\;-0.4\ln 10\;k_{\rm eff}\,(X_{\rm near}-X_{\rm far})\;+\;\varepsilon,
$$

where $\varepsilon$ is the gravity-wave fluctuation between the two lines of
sight — zero mean in the log and uncorrelated with the airmass difference. The
intrinsic emissivity cancels because both arms are the same coefficient at the
same instant. The recovered $k_{\rm eff}$ **absorbs the multiple-scattering
correction, the airglow-versus-stellar difference, the site aerosol level and
any residual error in the assumed layer height**, none of which has to be
modelled explicitly.

Four details make the estimator trustworthy:

- **Layer height is held fixed.** Because of the degeneracy above, only $k_{\rm eff}$ is fitted. Fitting both would be ill-conditioned.
- **Selection is at column level only.** A per-row amplitude threshold is
  selection on the outcome: whichever arm sits at lower airmass has the
  smaller van Rhijn factor and fails the cut unless it carries a positive
  fluctuation, which correlates the retained residual with the sign of
  $\Delta X$. In testing, a 40th-percentile per-row cut inflated the
  recovered $k_{\rm eff}$ by a factor of 1.8. Coefficients are instead kept
  or dropped **whole**, via `min_positive_fraction`; `retained_frac` is
  reported so residual row-level loss is visible.
- **Errors are cluster-robust, clustered on rows.** All coefficients in a
  row share one gravity-wave fluctuation, so naive OLS errors are far too
  small.
- **Stability is reported.** Each bin carries split-half values, and the
  fitted intercept — which should be zero. A significantly nonzero intercept
  indicates a relative throughput offset between the two sky channels rather
  than anything atmospheric.

`resolve_coef_extinction_k()` then interpolates the fitted values in
wavelength over well-constrained bins, falls back to the generic LCO stellar
curve elsewhere, and **clips into $[0,k_{\rm generic}]$**: the multiple-
scattering argument makes the stellar curve an upper bound, so a fitted value
above it signals noise or an unmodelled gradient, not physics. The resulting
per-coefficient array is stored on the dataset as `coef_extinction_k`,
threaded into `airglow_geometry_scale`, and saved in the training artifacts so
that prediction uses exactly the values training used. Set
`USE_FITTED_EXTINCTION = False` to revert to the generic curve.

## 2.6 Moon and zodi are exempt

Neither the moon nor the zodi group is a thin-shell emitter, so
`airglow_geometry_scale()` returns factor 1.0 for their coefficient columns.
The physics they need is instead:

- **moon**: the Krisciunas–Schaefer atmospheric scattering geometry. The
  ML head consumes `moon_alt`, `moon_phase`, `moon_sep`, `airmass`
  directly and learns the full non-linear dependence.
- **zodi**: the Leinert V-band lookup $B_{500}(\lambda_\odot^{\rm rel},|\beta_{\rm ecl}|)$
  times airmass. The ML head consumes ecliptic geometry, `airmass` and
  (via the isolated zodi branch, §3.4.1) `vanrhijn_285km` as a smooth
  F-region-height proxy for the extended zodiacal geometry.

Extinction on moon and zodi is not fitted here either. Aerosol variability
enters the moon amplitude directly (Krisciunas–Schaefer $k_{\rm ext}$), but
the same per-night degeneracy with height-of-scatter argument applies: the
absolute value cancels between sky and sci, and only the airmass difference
survives, which is at the per-cent level.

---

# Chapter 3 — ML Reconstruction & Prediction

With each row's sky-arm spectra now compressed into a physics-normalised
coefficient vector, the problem reduces to a mapping between coefficient
vectors: use the two sky arms' coefficients (plus row metadata) to predict
the science-arm's at the same instant.  Chapter 3 defines that mapping —
a symmetric dual-encoder neural network with per-group heads — and every
choice that hangs off it: the grouping into physically-meaningful blocks
(§3.2), the per-group compressors (§3.3), the isolated branches for the
anisotropic groups (§3.4.1–§3.4.3), the arm-blend prior (§3.5), and the
mixed flux/score-space loss (§3.6).

## 3.1 Goal

Given a row's paired coefficients from the two sky arms (near, far) and the
row's geometric / temporal context ("ctx"), predict the science-arm
coefficients $\hat{\mathbf{c}}_{\rm sci}$. Reconstructing
$\hat{y}_{\rm sci}(\lambda) = \mathbf{A}\hat{\mathbf{c}}_{\rm sci}$ then gives
the sky spectrum at the science pointing so that the science exposure can
be pixel-level sky-subtracted.

## 3.2 Coefficient grouping

Coefficients are routed into six physically meaningful groups by
`COEF_SCHEMA`:

| group          | patterns             | $n_{\rm coef}$ | dominant physics driver                        |
|:---------------|:---------------------|---------------:|:-----------------------------------------------|
| `mesospheric`  | `OH_\d{3}`, `O2_b\d+`|          358   | thermospheric temperature, geomagnetic activity |
| `moon`         | `Moon_bs\d+`         |             15 | $g_{\rm moon}(\phi, h, r_{\rm sep})$           |
| `zodi`         | `Zodi_bs\d+`         |              5 | $B_{500}(\lambda_\odot^{\rm rel}, |\beta_{\rm ecl}|)$ × airmass |
| `continuum`    | HO2, FeO, O2Ac       |              3 | mesospheric chemistry (residual)               |
| `atomic`       | ATOM_K               |              3 | night-side K I 7699                            |
| `ionospheric`  | ATOM_N               |              4 | night-glow N I 5199                            |

Deployed with the spline3 reduced basis (2026-08-25 changelog): $n_{\rm moon}=15$
from `n_spline_knots=11` and $n_{\rm zodi}=5$ from `n_zodi_spline_knots=1`. Total
$P = 388$ coefficients per row on `new-oh` (358 mesospheric).

`zodi` is promoted to its own group by this variant. The rationale is physical:
$g_{\rm moon}$ and $B_{500}$ depend on disjoint context sets (lunar geometry vs.
ecliptic geometry) and the previous baseline's ML head had to reconcile the two
on one output. Splitting the head lets each learn the projection appropriate to
its physical support.

## 3.3 Per-group compressor

Each group's raw coefficient block $\mathbf{c}_g\in\mathbb{R}^{n_g}$ is projected
into a compact "score" vector $\mathbf{s}_g\in\mathbb{R}^{n_g^{\rm score}}$ via
a fitted compressor consisting of three stages:

1. **Robust per-column scale** $\boldsymbol\rho_g$ = per-column median of
   positive training values on the near arm.
2. **Elementwise forward transform** $\mathbf{u}_g = f_{\rm kind}(\mathbf{c}_g/\boldsymbol\rho_g)$
   with `kind` chosen per group by `COMPRESSION_TRANSFORM_BY_GROUP`. The deployed
   defaults are `asinh` for `mesospheric` (coefficients span many decades and
   include OH lines close to zero) and `sqrt` for every other group (`moon`,
   `zodi`, `continuum`, `atomic`, `ionospheric`).
3. **Standardise + PCA truncation** on the pooled (near + far + sci) training
   scores:
   $\mathbf{s}_g = \big((\mathbf{u}_g - \boldsymbol\mu_g)/\boldsymbol\sigma_g\big)\,\mathbf{B}_g\big[:,\mathcal{K}_g\big]$,
   where $\mathbf{B}_g$ is the PCA basis (columns sorted by variance) and
   $\mathcal{K}_g$ retains components with **cross-arm correlation** above
   `COMPRESSION_XARM_THRESHOLD` on the held-out set. Groups with
   `COMPRESSION_USE_PCA_BY_GROUP[g]=False` skip PCA (identity basis).

Compressed target dimensions used in this variant:

- `moon`: 15 → 15 (PCA rotation; the retention threshold keeps all 15
  components, so the rotation is applied but nothing is dropped)
- `zodi`: 5 → 5 (no PCA)
- `mesospheric`: 403 → 403 (no PCA)
- `continuum`: 3 → 3 (no PCA)
- `atomic`: 3 → 3 (no PCA)
- `ionospheric`: 4 → 4 (no PCA)

Total compressed score dim = 433, matching the uncompressed count.

Inverse compressor:

$$
\hat{\mathbf{c}}_g \;=\; f_{\rm kind}^{-1}\!\big(\hat{\mathbf{s}}_g\,\mathbf{B}_g[:,\mathcal{K}_g]^\top\!\cdot\!\boldsymbol\sigma_g + \boldsymbol\mu_g\big)\odot\boldsymbol\rho_g,
$$

with $f_{\rm kind}^{-1}\in\{{\rm identity},\,x\mapsto x^2,\,\sinh,\,\exp\}$
clipped to the per-column training range to prevent $\sinh/\exp$ blow-up on
outlier predictions.

## 3.4 Architecture: symmetric dual-encoder with group heads

The network's task is to map two sky-arm coefficient vectors plus the row
context into the science-arm coefficient vector at the same instant.  Three
principles govern its architecture: shared encoders for the two sky arms
enforce arm-swap symmetry, per-group heads let physically distinct emitters
learn their own sky-to-sci transfer, and isolated branches route the
anisotropic groups (zodi, continuum) around the shared trunk so they see the
physical drivers the trunk squeezes out.

`DualEncoderGroupHeadMLPCompressed` is the deployed network. All widths and
hyperparameters live in `default_dual_group_config`; the values shown below
are defaults for this variant.

Notation: $\mathbf{s}^{\rm nr},\mathbf{s}^{\rm fr}\in\mathbb{R}^{n_s}$
compressed scores on the two sky arms (concatenated across groups,
$n_s\!\approx\!450$ after RobustScaler-scaling); $\mathbf{x}^{\rm nr},\mathbf{x}^{\rm fr},\mathbf{x}^{\rm sc}\in\mathbb{R}^{n_x}$
row context features per arm (astropy geometry, VanRhijn heights, solar
activity, etc.).

**Step 1 — per-arm score encoder** (shared weights, applied to each sky arm):

$$
\mathbf{e}^{\rm nr} = \mathrm{MLP}_{\rm sky}\!\big([\mathbf{s}^{\rm nr};\,\mathbf{x}^{\rm nr}_{\rm model}]\big),\qquad
\mathbf{e}^{\rm fr} = \mathrm{MLP}_{\rm sky}\!\big([\mathbf{s}^{\rm fr};\,\mathbf{x}^{\rm fr}_{\rm model}]\big).
$$

Widths: `encoder_dims=(768, 384)` (default) with LayerNorm + GELU
between layers.

**Step 2 — context encoder** on the science-arm ctx only:

$$
\mathbf{e}^{\rm ctx} = \mathrm{MLP}_{\rm ctx}\!\big(\mathbf{x}^{\rm sc}_{\rm model}\big),\qquad \text{widths } (64,).
$$

**Step 3 — symmetric fusion.** Combine the two arm embeddings into
symmetric summaries so that swapping near ↔ far leaves the network
invariant:

$$
\mathbf{e}^{\rm mean} = \tfrac{1}{2}(\mathbf{e}^{\rm nr}+\mathbf{e}^{\rm fr}),\quad
\mathbf{e}^{\rm diff} = \mathbf{e}^{\rm nr}-\mathbf{e}^{\rm fr},\quad
\mathbf{e}^{\rm |diff|} = |\mathbf{e}^{\rm diff}|.
$$

Concatenate with the context embedding:
$\mathbf{z} = [\mathbf{e}^{\rm mean};\,\mathbf{e}^{\rm diff};\,\mathbf{e}^{\rm |diff|};\,\mathbf{e}^{\rm ctx}]$.
The signed `e_diff` term lets the trunk see the sign of any asymmetry
(near closer to moon than far, or vice-versa), while `|e_diff|` gives a
sign-agnostic magnitude that the trunk can gate on regardless of orientation.

**Step 4 — shared trunk MLP**:

$$
\mathbf{h} = \mathrm{MLP}_{\rm trunk}(\mathbf{z}),\qquad \text{widths } (320, 160).
$$

**Step 5 — per-group heads.** Each group $g$ has its own head that produces
a predicted signed **score vector** $\hat{\mathbf{s}}^{\rm head}_g\in\mathbb{R}^{n_g^{\rm score}}$:

$$
\hat{\mathbf{s}}^{\rm head}_g = \mathrm{MLP}_{\rm head,\,g}(\mathbf{h}),\qquad \text{widths } (\text{head\_dim}=192,\, n_g^{\rm score}).
$$

Heads emit **linear** outputs (no Softplus) in scaled-score space; the
non-negativity of physical coefficients is recovered later by the inverse
compressor's clipping.

### 3.4.1 Isolated zodi branch

Zodiacal light is anisotropic on the ecliptic plane and its transfer from
sky to sci depends on which side of the ecliptic each pointing sits on.  The
shared trunk squeezes those geometric distinctions into a 160-d bottleneck
that mostly carries moon information; the isolated zodi head bypasses that
squeeze and reads a physics-motivated ctx subset directly.

When `default_dual_group_config['zodi_ctx_restriction']` is a non-empty tuple
of ctx feature names, the zodi head **bypasses the shared trunk** and consumes
only:

- the **per-arm slice** of the restricted ctx features (near + far + sci
  concatenated, so the branch sees the local `zodi_log10_v` gradient and any
  per-arm moon geometry),
- the near and far zodi score blocks (5 dims each in the deployed reduced-basis
  variant; see §1.4 / spline3).

Routed through a small two-layer MLP `zodi_branch: 64→32 (GELU)` followed by a
head emitting $n_{\rm zodi}^{\rm score}=5$
signed scores that match the 5 `Zodi_bs` knots after the spline3 basis reduction.

The deployed default restriction is 17 features (`sun_alt` and `alt` were
added 2026-09-01 as the top-ranked zodi residual predictors not already
routed to the branch):

```
('airmass', 'vanrhijn_285km',
 'ecl_beta_deg', 'ecl_lon_sin', 'ecl_lon_cos',
 'zodi_log10_v', 'sun_sep', 'sun_alt', 'alt',
 'moon_alt', 'moon_sep', 'moon_phase_sin', 'moon_phase_cos',
 'moon_fli', 'moon_up_smooth', 'moon_airmass_up', 'moon_signal_proxy')
```

giving $17 \times 3 + 2 \times 5 = 61$ input dims. Features cover two physical
roles:

1. **Zodi physics** &mdash; `ecl_beta_deg`, `ecl_lon_{sin,cos}`, `zodi_log10_v`,
   `sun_sep`, `airmass`, `vanrhijn_285km`. The Leinert lookup (§1.2.2) plus
   airmass (via the F-region-height van Rhijn as a smooth zodiacal-geometry
   proxy) fully determines the intrinsic zodiacal amplitude.
2. **Moon-driven contamination regime** &mdash; `moon_alt`, `moon_sep`,
   `moon_phase_{sin,cos}`, plus the four physics-prior moon-scatter proxies
   `moon_fli`, `moon_up_smooth`, `moon_airmass_up`, `moon_signal_proxy` (§2.6).
   On moon-down rows the `Zodi_bs` coefficients carry whatever residual
   continuum shape the `split_zodi` decomposition could not place elsewhere,
   so the sky↔sci transfer depends on moon geometry even though the underlying
   zodiacal light does not. Adding the moon proxies lets the branch condition
   on "moon down" / "Q4 phase" / "bright moon close-in" regimes; combined with
   the per-regime Jensen lift (§3.6.6) the residual zodi bias on `moon_alt ≤ 0`
   rows fell from 7.7% to ~2% as the isolated branches were added.

The previous `sun_alt` entry (twilight regime) was dropped when
the physics-prior augment absorbed its role via `moon_signal_proxy`, which
already vanishes when the moon is down and lets the moon-brightness features
carry the twilight-residual signal.

Bypassing the trunk cuts moon-driven and geomagnetic-driven leakage into the
zodi head at the cost of the trunk's richer representation. Empirically this
trade-off wins for zodi calibration on high-$|\beta_{\rm ecl}|$ / bright-moon
rows without hurting overall mean_eRMSE.

### 3.4.2 Isolated continuum branch

Mirroring the zodi pattern, when `continuum_ctx_restriction` is a non-empty
tuple the continuum head **bypasses the shared trunk** through its own branch.
The trigger for this was a specific empirical finding: the
`residual_ctx_attribution` diagnostic (5-fold CV random-forest fit of per-row
per-group residual RMS on ctx) surfaced continuum with rf_R² = 0.70 driven
entirely by moon geometry — a non-linear coupling the shared 160-d trunk was
under-parameterised to represent.

The deployed default restriction is 9 features:

```
('moon_alt', 'moon_sep', 'moon_phase_sin', 'moon_phase_cos',
 'moon_fli', 'airmass',
 'moon_fli_x_phase_cos', 'moon_sig_x_lon_cos', 'moon_sig_x_lon_sin')
```

The last three are explicit interaction features (§3.4.5):
`moon_fli * moon_phase_cos` (2nd-order phase term) and
`moon_signal_proxy * ecl_lon_{cos,sin}` (moon-scatter × zodi anisotropy).

Branch: `continuum_branch: 128→64 (GELU)` (widened from `64→32` — the only
group with real headroom above the sky-arm noise floor
got the extra capacity), followed by a `(64,)`-hidden head emitting
$n_{\rm continuum}^{\rm score}=3$ scores. Input dim: $9 \times 3 + 2 \times 3
= 33$. Isolated-branch parameter cost: ~16k.

### 3.4.3 Additive moon-zodi coupling

Both moon (via the shared-trunk head) and zodi (via the isolated branch)
benefit from knowing the *joint* moon-scatter + zodi-geometry state — scattered
moonlight adds to the diffuse ecliptic background that the split-zodi
decomposition tries to route entirely into `Zodi_bs`. To let the two heads
*share* that joint representation without breaking the zodi branch's isolation,
an **additive coupling latent** is threaded into both head outputs.

When `moon_zodi_ctx_restriction` is a non-empty tuple (deployed default), the
model builds:

- a small `moon_zodi_coupling_branch: 94→64→32 (GELU)` (~8.4k params),
- a **zero-initialised** projector
  $P_{\rm moon} = \mathrm{Linear}(32,\, n_{\rm moon}^{\rm score}=15)$ (495 params),
- a **zero-initialised** projector
  $P_{\rm zodi} = \mathrm{Linear}(32,\, n_{\rm zodi}^{\rm score}=5)$ (165 params).

The branch consumes the union of moon-scatter + zodi-geometry ctx: 18 features
$\times$ 3 arms plus 2 arms $\times (n_{\rm moon}+n_{\rm zodi}) = 20$ score
dims, for 94 input dims total.

The forward pass then becomes (for $g \in \{{\rm moon},\,{\rm zodi}\}$):

$$
\hat{\mathbf{s}}_g \;=\; \alpha_g\,\mathbf{s}^{\rm nr}_g + (1-\alpha_g)\,\mathbf{s}^{\rm fr}_g \;+\; \hat{\mathbf{s}}^{\rm head}_g \;+\; P_g\bigl(\mathrm{CouplingBranch}(x_{\rm mz})\bigr),
$$

where $\hat{\mathbf{s}}^{\rm head}_g$ is the moon shared-trunk head output
$\mathrm{head}_{\rm moon}(\mathbf{h})$ for $g={\rm moon}$ and the isolated zodi
branch output $\mathrm{ZodiHead}(\mathrm{ZodiBranch}(x_{\rm zodi}))$ for
$g={\rm zodi}$. Because the projectors are initialised to output exactly zero
(both weights and bias zero), **training starts byte-identical to the
uncoupled configuration**; the projectors learn what fraction of the
coupling latent to route into each head's residual.

The `moon_zodi_ctx_restriction` 18-feature union is the superset of the zodi
restriction (§3.4.1) plus the 3 interaction features:

```
('airmass', 'vanrhijn_285km',
 'ecl_beta_deg', 'ecl_lon_sin', 'ecl_lon_cos',
 'zodi_log10_v', 'sun_sep',
 'moon_alt', 'moon_sep', 'moon_phase_sin', 'moon_phase_cos',
 'moon_fli', 'moon_up_smooth', 'moon_airmass_up', 'moon_signal_proxy',
 'moon_fli_x_phase_cos', 'moon_sig_x_lon_cos', 'moon_sig_x_lon_sin')
```

An earlier alternate that
retired both the shared-trunk moon head AND the isolated zodi branch and
routed moon+zodi through a single larger branch was tried and reverted — it
captured a bigger moon improvement (~7% moon_up sky-arm) but reintroduced the
moon→zodi contamination and regressed atlas mid-band RMS by 35%. The
mode selector and its code path were removed from the trainer in a dead-code
sweep; only the additive coupling remains.

### 3.4.4 Data-flow diagram of the augmented model

```
    near-arm scores   far-arm scores   near-ctx (39)   far-ctx (39)   sci-ctx (39)
          |                 |               |               |             |
          +--- concat ------+               v               v             |
          |    scores/ctx                   |               |             |
          v                                 |               |             |
    +-----------+                       +----------------+                |
    | sky enc.  |                       |  sky encoder   |                |
    | (shared)  |------e_near-----+     |   (shared)     |------e_far     |
    +-----------+                 |     +----------------+          |     |
                                  v                                 v     v
                        e_mean = (e_near+e_far)/2  e_diff = e_near-e_far  |
                        |e_diff|                                          |
                        |                                                 |
                        v                                                 v
                        +---------- concat with e_ctx ---------------------+
                                          |
                                          v ctx encoder(sci) -> e_ctx
                                          |
                                    +-----------+
                                    |   trunk   |  (320 -> 160)
                                    +-----+-----+
                                          |
                                          v h (160-d)
                             +--------+---+---+--------+-----------------+
                             |        |       |        |                 |
                             v        v       v        v                 |
                         moon head  meso    iono     atomic              |
                                    head    head     head                |
                                    (shared trunk consumers)             |
                                                                         |
                        +------------------------------------------------+
                        |                                                |
                        v                                                v
                +---------------+                              +----------------+
                | zodi branch   |  (55-d in)                   | continuum br.  |
                | 64 -> 32      |                              | 128 -> 64      |
                | + zodi head   |                              | + cont. head   |
                | (32,) + n_z=5 |                              | (64,) + n_c=3  |
                +-------+-------+                              +--------+-------+
                        |                                              |
        (additive coupling latent, applied to moon + zodi)              |
        +----------------+                                              |
        |                                                               |
        v                                                               v
+-------------------+                                                +----+
| coupling branch   |                                                |    |
| 94 -> 64 -> 32    |                                                |    |
+-------+-----------+                                                |    |
        |                                                            |    |
        +--> P_moon = Linear(32, 15), zero-init  --> + moon head      |    |
        +--> P_zodi = Linear(32,  5), zero-init  --> + zodi head      |    |
                                                                     v    |
                                                              +---+ +---+ +---+ +---+ +---+
                                                     out[g] = |bl.|+|hd |+|D_g|  for each  g
                                                              +---+ +---+ +---+     of six
                                                    where D_g = coupling residual (moon,
                                                    zodi only) or 0 for the other four.

Blend for group g (see §3.5):
    hat{s}_g = alpha_g * s_g_near + (1 - alpha_g) * s_g_far + head_g + P_g(coupling)
    alpha_g is scalar OR per-row sigmoid(w.x_sci + b) for anisotropic
    groups (moon, zodi, continuum) via the ctx-alpha predictor.
```

A row's forward pass proceeds in five stages.  Each sky arm's compressed score
vector is concatenated with its 39-dimensional context and passed through the
**shared sky encoder** (widths $768\to 384$); tying the weights across the two
arms produces a single per-fibre embedding that is invariant to which physical
arm produced the input.  The two arm embeddings are then combined into three
exchange-symmetric summaries — the mean $\mathbf{e}^{\rm mean}$, the signed
difference $\mathbf{e}^{\rm diff}$, and its absolute value
$|\mathbf{e}^{\rm diff}|$ — so that the trunk sees both the direction and the
magnitude of any sky-to-sky asymmetry while remaining invariant under a
relabelling of the arms.  The science-arm context is embedded separately by a
compact **context encoder** (width 96), concatenated with the arm summaries,
and compressed by the **trunk MLP** ($320\to 160$) to a 160-dimensional latent
$\mathbf{h}$ that carries the row's shared geometric and temporal state.

Six per-group heads consume the pipeline outputs.  Four of the six — moon,
mesospheric, ionospheric, atomic — read $\mathbf{h}$ directly through a
two-layer head (width $192\to n_g^{\rm score}$).  The physically anisotropic
zodi and continuum groups instead route through **isolated branches**
(§3.4.1, §3.4.2), whose inputs are three-arm slices of a physics-motivated
context subset stacked with the two arms' native score blocks.  Bypassing the
trunk gives these two groups a direct view of the physical drivers — ecliptic
geometry for zodi, moon-scatter geometry for continuum — that the shared
representation squeezes out.

Moon and zodi are additively coupled by a small **coupling branch** (§3.4.3)
that runs once per batch on the union of moon-scatter and zodi-geometry
context; its 32-dimensional latent is projected into each of the two heads by
zero-initialised linear maps, so at $t{=}0$ the coupling contribution is
identically zero and training reproduces the uncoupled baseline byte for
byte.  During training the projectors learn what fraction of the joint
moon-zodi state to route into each head's residual without disturbing zodi's
isolated-branch representation; the four non-moon-non-zodi heads receive an
identically-zero coupling term by construction.  Each group's final score is
then an arm blend (§3.5) plus the head residual plus — for moon and zodi
alone — the coupling projection.

### 3.4.5 Context feature routing

The context vector is **39 features per arm** after the two augment stages
(ECLIPTIC-CTX-V1 and PHYSICS-PRIORS-V1). Every feature enters the sky encoder
(per-arm) and the ctx encoder (sci-arm), so the shared trunk sees the full
ctx. The specialised branches and the ctx-α predictor take restricted slices.

**Feature categories** (39 total):

| category | count | features |
|---|---:|---|
| Sci-pointing astrometry | 4 | `alt`, `az_sin`, `az_cos`, `airmass` |
| Moon geometry | 6 | `moon_alt`, `moon_sep`, `moon_phase_{sin,cos}`, `moon_az_{sin,cos}` |
| Sun geometry | 4 | `sun_sep`, `sun_alt`, `sun_az_{sin,cos}` |
| Sci-arm separation | 1 | `sci_sep` |
| Van Rhijn slant-path factors | 3 | `vanrhijn_87km`, `vanrhijn_95km`, `vanrhijn_285km` |
| Cyclic time features | 6 | `obstime_{day,lunation,year}_{sin,cos}` |
| Space-weather activity | 4 | `f107`, `f107_81d`, `kp`, `ew` |
| Ecliptic geometry (ECLIPTIC-CTX-V1) | 3 | `ecl_beta_deg`, `ecl_lon_{sin,cos}` |
| Physics-prior moon-scatter proxies (PHYSICS-PRIORS-V1) | 5 | `zodi_log10_v`, `moon_fli`, `moon_up_smooth`, `moon_airmass_up`, `moon_signal_proxy` |
| explicit interaction features | 3 | `moon_fli_x_phase_cos`, `moon_sig_x_lon_{cos,sin}` |

**Feature-to-branch routing** (✓ = feature enters that branch as one of its
ctx dims; empty = does not). The **shared trunk** column stays implicit —
every feature feeds the trunk via the encoders.

| feature | isolated zodi (§3.4.1) | isolated continuum (§3.4.2) | moon-zodi coupling (§3.4.3) | ctx-α (§3.5) |
|---|:---:|:---:|:---:|:---:|
| `alt`, `az_sin`, `az_cos` | | | | |
| `airmass` | ✓ | ✓ | ✓ | ✓ |
| `moon_alt`, `moon_sep` | ✓ | ✓ | ✓ | |
| `moon_phase_sin`, `moon_phase_cos` | ✓ | ✓ | ✓ | |
| `moon_az_sin`, `moon_az_cos` | | | | |
| `sun_alt`, `sun_az_{sin,cos}` | | | | |
| `sun_sep` | ✓ | | ✓ | |
| `sci_sep` | | | | |
| `vanrhijn_87km`, `vanrhijn_95km` | | | | |
| `vanrhijn_285km` | ✓ | | ✓ | |
| `obstime_{day,lunation,year}_{sin,cos}` | | | | |
| `f107`, `f107_81d`, `kp`, `ew` | | | | |
| `ecl_beta_deg` | ✓ | | ✓ | ✓ |
| `ecl_lon_sin`, `ecl_lon_cos` | ✓ | | ✓ | |
| `zodi_log10_v` | ✓ | | ✓ | |
| `moon_fli` | ✓ | ✓ | ✓ | |
| `moon_up_smooth` | ✓ | | ✓ | ✓ |
| `moon_airmass_up`, `moon_signal_proxy` | ✓ | | ✓ | |
| `moon_fli_x_phase_cos` | | ✓ | ✓ | |
| `moon_sig_x_lon_cos`, `moon_sig_x_lon_sin` | | ✓ | ✓ | |

**Head-to-branch routing** (which head consumes which upstream output):

| head | shared trunk | isolated zodi | isolated continuum | moon-zodi coupling residual |
|---|:---:|:---:|:---:|:---:|
| `moon` | ✓ | | | ✓ (additive) |
| `zodi` | | ✓ | | ✓ (additive) |
| `continuum` | | | ✓ | |
| `mesospheric` | ✓ | | | |
| `ionospheric` | ✓ | | | |
| `atomic` | ✓ | | | |

The specialised branches were added when the diagnostics pointed at
regime-specific structure the shared trunk was under-fitting; the additive
coupling was added to let moon share the zodi branch's ctx-restricted
representation without breaking zodi's isolation. See the changelog entries
for the empirical wins each branch bought.

## 3.5 Blend heads

Even before the network learns anything, a good first guess for the science
coefficients is a linear interpolation of the two sky arms.  The blend heads
give the network that starting point as an explicit prior and have each head
predict a residual on top; at $\alpha_g = 0.5$ and zero head output the model
returns the arm mean, which is where training begins.

Each group carries a **blend weight** $\alpha_g \in (0, 1)$ that mixes the
near/far arm scores with the head's residual prediction:

$$
\boxed{\;\hat{\mathbf{s}}_g \;=\; \alpha_g\,\mathbf{s}^{\rm nr}_g + (1-\alpha_g)\,\mathbf{s}^{\rm fr}_g \;+\; \hat{\mathbf{s}}^{\rm head}_g \;+\; \Delta_g^{\rm coupling}.\;}
$$

The additive coupling residual $\Delta_g^{\rm coupling}$ (§3.4.3) is non-zero
only for $g \in \{{\rm moon},\,{\rm zodi}\}$ when the coupling branch is
active; it is exactly zero at initialisation and remains identically zero for
the other four groups.

**Two parametrisations for $\alpha_g$:**

1. **Scalar α** (fallback): $\alpha_g$ stored directly (not through a
   sigmoid) as a single learnable scalar per group, clamped to
   $[\varepsilon,\, 1-\varepsilon]$ with $\varepsilon = 10^{-3}$ after each
   optimiser step. This is what the physically isotropic groups still use.

2. **Context-dependent α** (deployed for the
   anisotropic groups): when `alpha_ctx_features` is a non-empty tuple of
   ctx feature names, each group in the model's `_ALPHA_CTX_GROUPS` gets a
   **per-row per-group** linear predictor
   $$\alpha_g(x_{\rm sci}) \;=\; \sigma\bigl(\mathbf{w}_g \cdot \mathbf{x}_{\rm sci}^{\alpha} + b_g\bigr),$$
   where $\mathbf{x}_{\rm sci}^{\alpha}$ is the sci-arm slice of the requested
   features. The linear predictor's weights are initialised to zero and its
   bias to $\mathrm{logit}(\alpha_{\rm init})$ so that at $t=0$ every row
   gets $\alpha_g = \alpha_{\rm init}$ (default 0.7), preserving byte-identity
   with the scalar-α starting point.

**Deployed defaults:**

- `alpha_ctx_features = ('moon_up_smooth', 'ecl_beta_deg', 'airmass')` — three
  features that carry moon-brightness, zodi-anisotropy, and airmass geometry,
  so the anisotropic groups can learn distinct blend rules per regime.
- `_ALPHA_CTX_GROUPS = ('moon', 'zodi', 'continuum')` (hardcoded in
  `model.py`, not a config knob) — restricted to the
  physically anisotropic groups. The LOS-integrated thin-shell groups
  (`mesospheric`, `ionospheric`, `atomic`) keep the scalar α near 0.5: their
  sky↔sci transfer is genuinely isotropic on the LVM scale, and an early
  experiment applying ctx-α to all six groups blew up ionospheric sRMSE by
  23% before 26d restricted it.

**α does not converge on its own** (2026-09-01). At the default learning
rate the loss surface in α is nearly flat, because the additive head absorbs
the *systematic* part of a mis-set blend — but not the per-row arm noise the
blend passes straight through, so a wrong α costs real accuracy while
generating almost no gradient to correct itself. The ctx path is damped a
further ~4× by the sigmoid ($\sigma' = 0.25$ at $\alpha=0.5$, $0.128$ at
$0.85$) and by splitting the gradient across three zero-init weights, and
early stopping at epochs 11–29 truncates the climb. Measured end-of-training
α from otherwise identical runs differing only in init:

| group | init 0.5 | init 0.7 | init 0.85 |
|:---|---:|---:|---:|
| `mesospheric` | 0.685 | 0.774 | 0.789 |
| `ionospheric` | 0.634 | 0.800 | 0.947 |
| `atomic`      | 0.735 | 0.838 | 0.927 |
| `moon` (ctx-α)      | 0.533 | 0.729 | 0.859 |
| `zodi` (ctx-α)      | 0.522 | 0.715 | 0.857 |
| `continuum` (ctx-α) | 0.553 | 0.734 | 0.862 |

`blend_init_alpha` therefore behaves as a **hyperparameter, not an
initialisation**, unless `alpha_lr_mult` (§3.7) is raised.

**Measured optima** at `blend_init_alpha=0.85`, `alpha_lr_mult=30`, where α
does travel: `mesospheric` ≈ 0.79 (bracketed — it rises to 0.774 from below
and falls to 0.789 from above), `ionospheric` ≈ 0.95, `atomic` ≈ 0.93, and the
ctx-α groups ≈ 0.86–0.91 with per-row spans of [0.59, 0.96] for `moon` and
[0.51, 0.99] for `continuum`.

Two predictions in earlier versions of this section were **wrong**:

- `zodi` was expected to sit near 0.5 as an isotropic emitter. Measured
  α ≈ 0.895, spanning [0.79, 0.95] — zodi strongly prefers the near arm,
  consistent with `Zodi_bs` absorbing residual continuum that is not
  zodiacal light.
- Every group prefers the near arm over the arm mean, which `naive_baseline`
  had always shown directly: `B0_copy_near` beats `B2_mean_geo` for all six
  groups, ionospheric by 2.7× (5.18 vs 13.96). Setting α to a "neutral" 0.5
  is a large regression, not a variance-minimising default.

The ctx-α predictor reads only the **science-arm** ctx slice, so it cannot
express *which* arm is better placed (that is a `near_ctx − far_ctx`
question). It still learns useful per-row regime structure once it can
travel, but the arm-relative information remains unavailable to it.

## 3.6 Loss

The training loss is a **mixed-space** objective. The moon and zodi groups are
trained in **flux space** — the network's predicted compressed scores are
decompressed all the way to per-pixel spectra inside the loss and compared to
the target flux — while the other four groups (continuum, mesospheric,
ionospheric, atomic) stay in **scaled-score space** so that RobustScaler-
normalised residuals are comparable across compressed groups of very different
physical amplitudes. §3.6.1 defines the flux-space branch. §3.6.2 defines the
score-space branch. §3.6.3 covers the per-element COEF_ERR weighting.
§3.6.4 the diagnostic-only pixel WRMSE. §3.6.5 the row-weight boosts.
§3.6.6 the post-training per-coef Jensen-style lift.

### 3.6.1 Per-pixel flux MSE on moon and zodi (deployed)

For each group $g \in$ `flux_mse_groups = ('moon', 'zodi')` (deployed default)
the training loss is computed **directly on the reconstructed per-pixel
spectra**, not on the compressed score residuals. The motivation is that the
diagonal COEF_ERR-weighted score loss (§3.6.2 + §3.6.3) puts per-row weight
$\propto 1/\sigma^2$, and the truth-conditioned $|z|$ diagnostic (§1.9) shows
$\sigma_{\rm scaled}$ is essentially flat across the moon-brightness axis:
bright-moon rows (with $|c_{\rm moon}| \sim 10$–20, where flux fit errors
dominate the deliverable sky subtraction) end up on the same per-row loss
scale as moon-down rows (with $|c_{\rm moon}| \sim 0.05$). A flux-space MSE
couples per-row weight to target amplitude directly: the same absolute
score-space error contributes $\sim (|c_{\rm bright}|/|c_{\rm dim}|)^2 \sim
10^5$ times more from a bright row than from a dim row, matching the ratio
at which they matter in flux.

**Inverse compressor in torch.** For each configured group $g$ the trainer
maintains a differentiable torch implementation of `inverse_group_compressor`
(§3.3):

1. Undo the RobustScaler on the group's compressed score
   $\hat{\mathbf{s}}_g^{\rm scaled}$:
   $\hat{\mathbf{s}}_g^{\rm raw} = \hat{\mathbf{s}}_g^{\rm scaled}\odot \mathrm{scale}_g + \mathrm{center}_g$.
2. Undo the group's PCA rotation:
   $\mathbf{z}_g = \hat{\mathbf{s}}_g^{\rm raw} \, B_g^\top$
   ($B_g$ = identity when `use_pca=False`, e.g. zodi; a small orthogonal
   matrix stored on device when `use_pca=True`, e.g. moon).
3. Destandardise to forward-transformed space:
   $\mathbf{y}_g = \mathbf{z}_g \odot \mathrm{sd\_vec}_g + \mathrm{mean\_vec}_g$.
4. Invert the per-element transform. Both moon and zodi use `kind='sqrt'` so
   $\mathbf{em}_g = \max(\mathbf{y}_g, 0)^2$
   (sign is intentionally lost by the forward $\sqrt{}$; both target and
   prediction go through the same clamp so gradients are consistent).
5. Restore per-row geometry:
   $\hat{\mathbf{c}}_g^{(r)} = \mathbf{em}_g \odot \mathbf{sc}_{\rm sci}^{(r)}$
   where $\mathbf{sc}_{\rm sci}^{(r)}$ is the per-row per-coefficient geometry
   factor from `airglow_geometry_scale(ctx_sci, ...)` (see §2).

The identical chain is applied to the target scaled-score
$\mathbf{s}_g^{{\rm true},{\rm scaled}}$ so that both paths are in the same
physical units.

**Flux basis matrix.** A row-independent basis matrix
$A_g \in \mathbb{R}^{n_g^{\rm coef} \times n_\lambda^{\rm ds}}$
is precomputed once, before the ensemble seed loop, by instantiating

```python
SkyDecompLSFSurfaceIterative(
    wave_ref, lsf_sigma=1.0, n_spline_knots=N_MOON_KNOTS,
    split_zodi=SPLIT_ZODI, n_zodi_spline_knots=N_ZODI_KNOTS,
    base_dir=_infer_base_dir_for_reconstruction(),
)
```

on the notebook's wavelength grid. The `SkyDecomp.__init__` build path (§1.2)
populates `matrix_moon` and `matrix_zodi` — the pre-LSF spline template
matrices $B_k(\lambda) \cdot \mathrm{solar\_rb}(\lambda) \cdot \mathrm{envelope}_g(\lambda)$,
one row per moon or zodi knot. The wavelength axis is decimated with stride 5
(a smoothly-varying continuum basis is invariant to this;
$n_\lambda^{\rm ds} \approx 2\,500$) which brings the per-batch matmul into a
comfortable GPU footprint while keeping the moon-albedo mineral bands and the
zodi power-law tilt at full resolution.

**Per-row loss.** With $A_g$ on device, the per-row per-group loss is

$$
\mathcal{L}_g^{\rm flux,(r)} \;=\; s_g \cdot \frac{1}{n_\lambda^{\rm ds}} \sum_{\lambda=1}^{n_\lambda^{\rm ds}} \Big( \hat{\mathbf{c}}_g^{(r)} A_g - \mathbf{c}_g^{{\rm true},(r)} A_g \Big)^2_\lambda
$$

and the group is added into the total loss with the same balancing weight
$w_g = m_g / \sqrt{n_g^{\rm score}}$ (§3.6.2) as the score-space groups, so
`moon_group_weight` and `zodi_group_weight` retain the calibrated meaning they
had under the score-space loss.

The per-group **scale-match factor**

$$
s_g \;=\; \frac{\text{median}_{r \in \text{train}}\; L_g^{\rm diag,(r)}}{\text{median}_{r \in \text{train}}\; \overline{f_g^{{\rm true},(r)}(\lambda)^2}}
$$

is computed once at trainer entry: it aligns the median per-row flux-MSE with
the median per-row diagonal-Huber loss (§3.6.2), so `moon_group_weight` and
`zodi_group_weight` keep the balancing meaning they would have under the
score-space form.

**It replaces, not augments.** For a group in `flux_mse_groups` the
flux-space term *substitutes* for that group's coefficient-space
`smooth_l1` (an `if`/`else` in `compressed_loss`), so moon and zodi are scored
in flux units while the remaining four groups are scored in coefficient units.
The term is also strictly per-group — each group's own coefficients through its
own basis — so no moon+zodi **sum** is constrained anywhere.

**Measured effect** (10-seed ensembles on the identical split, gain over
`B0_copy_near`, all-regime):

| `flux_mse_groups` | moon | zodi | continuum | mesospheric |
|---|---|---|---|---|
| `('moon','zodi')` (deployed) | +24.0% | **+41.3%** | +19.8% | +6.7% |
| `('moon',)` | +22.6% | +31.6% | +19.1% | +7.4% |
| `('zodi',)` | +3.0% | +27.6% | +6.8% | +4.9% |
| `()` | +5.5% | +23.2% | +12.7% | +2.3% |

Three things follow. The moon needs *its own* flux term — a 20-point cliff
that tracks only whether moon is in the list, and giving the term to zodi
instead does the moon no good at all. The effect is **not separable**: zodi
reaches +41.3% only when both are present, and is better served by the moon's
term (+31.6%) than by its own (+27.6%). And continuum, which has no flux term,
tracks the moon arm exactly (19.8/19.1 with moon on, 12.7/6.8 with it off) —
the moon head sits on the shared trunk, so scoring it in flux space shapes the
representation every other group reads. Dropping the term entirely also
triples the seed-to-seed spread (0.28 → 0.81).

Consequently `zodi`-only is the worst configuration tested, worse than no flux
term at all on continuum and moon: giving the 5-coefficient group a flux term
while the 15-coefficient moon group loses one lets the zodi gradients dominate
a trunk that no longer receives moon-flux structure.


### 3.6.2 Weighted MSE on the score-space groups

For $g \in$ {`continuum`, `mesospheric`, `ionospheric`, `atomic`}, let
$\hat{\mathbf{s}}_g^{(r)},\,\mathbf{s}_g^{{\rm true},(r)}\in\mathbb{R}^{n_g^{\rm score}}$
be the predicted and target scaled-scores for row $r$ and group $g$. The
per-row per-group loss is

$$
\mathcal{L}_g^{(r)} \;=\; \frac{w_g}{n_g^{\rm score}}\,\sum_{k=1}^{n_g^{\rm score}} w^{\rm pe}_{g,k,r}\,\big(\hat{s}_{g,k,r} - s^{\rm true}_{g,k,r}\big)^2,
$$

with $w_g$ the **per-group balancing weight**

$$
w_g \;=\; \frac{m_g}{\sqrt{n_g^{\rm score}}},
$$

where the multipliers $m_g$ are the current deployed defaults:

- $m_{\rm moon}=3.5,\; m_{\rm zodi}=2.0,\; m_{\rm continuum}=1.0,\; m_{\rm mesospheric}=1.0,\; m_{\rm ionospheric}=1.0$.

Two 2026-09-01 A/B results constrain these. Raising $m_{\rm mesospheric}$ to
1.75 **degraded all six groups, mesospheric included**, and doubled the seed
std: that group is noise-limited, not capacity-starved (§3.8). Lowering
$m_{\rm moon}$ back to 2.0 was neutral on moon err/$\Delta$ but *doubled*
moon's NIR self-bias — because moon trains in flux space (§3.6.1), its weight
is what restrains red-end flux drift, so the elevated value is load-bearing.

The $1/\sqrt{n_g^{\rm score}}$ factor makes the sum-over-$k$ RMS-scale
comparable across compressed groups with different score dimensionality. Moon
and zodi carry $m_g=2.0$ multipliers even though their loss is computed in
flux space (§3.6.1) — the scale-match factor ports the multiplier meaning
across.

### 3.6.3 Heteroscedastic per-element weights

The decomposition-side coefficient uncertainties $\boldsymbol\sigma_{\rm coef,sci}$
(§1.9) are propagated through the compressor with a first-order Jacobian:

$$
\sigma_{s_{g,k,r}} \;=\; \big\lVert \nabla_{c_g} s_{g,k} \big\rVert \cdot \sigma_{{\rm coef},g,r},
$$

divided by the RobustScaler column scale so the weights live in the same space
as $\hat{\mathbf{s}}_g$. The per-element weight is

$$
w^{\rm pe}_{g,k,r} \;=\; \frac{1}{\sigma_{s_{g,k,r}}^2},\qquad \sigma_{s_{g,k,r}} \ge \sigma^{\rm floor}_{g,k},
$$

with $\sigma^{\rm floor}_{g,k}$ = `coef_err_sigma_floor_rel[g]` × per-column
median finite sigma. The floors are 5% for moon, zodi, continuum, and
ionospheric; 20% for mesospheric and atomic, because their $p_{99}/p_{50}$
sigma ratios span $10^{3}$–$10^{6}$ (see §7 diagnostic). Weights are
normalised so $\mathbb{E}_{\rm train}[w^{\rm pe}]=1$ per column; missing/
boundary sigmas fall through to the floor.

**Joint covariance diagnostic path.** The `coef-cov-loader` notebook cell
opens each of the three decomp FITS products, reads the `COEF_COV_MOON` and
`COEF_COV_ZODI` HDUs when present, slices them by `filtered_triplet['row_index']`,
and attaches `coef_cov_{moon,zodi}_{near,far,sci}` to `filtered_triplet`.
Two diagnostic cells consume these:

- `truth_conditioned_sigma_calibration` — computes the Mahalanobis residual
  $|z|_{\rm joint,g} = \sqrt{\mathbf{r}_g^{\top}\mathbf{\Sigma}_g^{-1}\mathbf{r}_g / n_{\rm active}}$
  per test row and reports its median vs the calibrated target
  $\sim\sqrt{\text{med}(\chi^2_n)/n}\approx 0.7$.
- `moon_sigma_investigation` — reports the joint per-block SNR
  $\sqrt{\mathbf{c}_g^{\top}\mathbf{\Sigma}_g^{-1}\mathbf{c}_g / n_{\rm active}}$
  binned by moon regime, alongside the independence-assuming
  $\lVert\mathbf{c}_g\rVert / \sqrt{\sum_j\sigma_j^2}$ for direct comparison.

These are diagnostic-only: the deployed loss uses only the diagonal
per-element $w^{\rm pe}$ weight above.

### 3.6.4 Physics-space pixel WRMSE (diagnostic only)

Additionally the notebook tracks a **physics-space** metric per row:

$$
{\rm WRMSE}^{(r)} \;=\; \sqrt{\frac{1}{N_\lambda}\sum_\lambda \big(\hat{y}_r(\lambda)-y_r(\lambda)\big)^2 / \sigma_{\rm tot}^2(\lambda)}
$$

with $\sigma_{\rm tot}$ from `FLUX_SIGMA_TOTAL`. This is used for diagnostic
reporting (worst-row visualisations, extrapolation quality `eRMSE`, per-
lunation drift) but is **not** part of the training gradient. The training
loss is §3.6.1 (moon and zodi) plus §3.6.2 on the remaining four groups.

### 3.6.5 Row weights

Row weights are **uniform**: $w_{\rm row}=1$ for every row. Every per-row
reweighting scheme tried was rejected on measurement — a high-airmass boost
and a moon-down-ecliptic boost never helped, and a bright-moon-close boost
(FLI $\ge$ 0.90, separation $\le 30°$, moon up, ×1.5) leaked into the blue
band of the residual atlas (+23% RMS|frac|) for a mid-band gain the
flux-space loss now supplies directly. The mechanism is not a config knob:
carrying it as one left a silent degree of freedom in the loss that had to be
re-checked on every run.

### 3.6.6 Post-training per-coef Jensen-style lift calibration

After the ensemble is assembled, a **single-pass empirical bias correction**
is computed on the train+val subset and stored in `jensen_corrections`,
applied at inference in `expand_scores_to_coefs`. Three flavours:

- **`moon` (per-coef vector)**: for each of the 15 `Moon_bs` knots the ratio
  $\ell_k = \bar{c}^{\rm true}_k / \bar{c}^{\rm pred,\,naive}_k$ is computed
  from calibration-row means, ignoring near-zero knots
  ($|\bar{c}^{\rm pred}| < 0.05|\bar{c}^{\rm true}|$) which default to
  $\ell_k=1$. Clipped to $[0.7, 1.4]$ to protect against a broken knot
  cascading. The moon per-coef lift nulls the spectral tilt bias that a
  single scalar cannot correct.
- **`zodi` (per-coef vector, regime-conditioned)**: three moon-alt regime
  buckets are lifted independently (`moon_up` for `moon_alt > 10°`,
  `moon_horizon` for `-10° \le moon_alt \le 10°`, `moon_down` for
  `moon_alt < -10°`) with a 5°-wide smooth boundary. Each bucket uses the
  same per-coef ratio $\ell_k$ formula on its subset; if a bucket has fewer
  than 30 calibration rows it falls back to the global per-coef lift.
  Clipped to $[0.7, 1.4]$. At inference the row's `moon_alt` selects the
  bucket (smoothed at the boundaries).
- **`continuum`, `mesospheric`, `ionospheric`, `atomic` (scalar)**:
  $\ell = \bar{c}^{\rm true}/\bar{c}^{\rm pred,\,naive}$, clipped to
  $[0.5, 2.0]$. Skipped if the mean is degenerate or the relative magnitude
  is below 5%.

At inference each predicted coef vector for group $g$ is multiplied by
$\boldsymbol\ell_g$ (per-coef for moon, regime-conditioned per-coef for zodi,
broadcast scalar for the others) inside `expand_scores_to_coefs`. This closes
the residual coherent bias that the network's own outputs leave in place
without adding trainable parameters.

## 3.7 Optimisation and splits

Training runs are deliberately conservative: a fixed AdamW schedule, early
stopping on a night-held-out validation set, and a 10-seed ensemble whose
mean is the deployed prediction.  The interesting design work is upstream in
§3.3–§3.6; §3.7 records the training-loop settings for reproducibility.

- **Optimiser**: AdamW, `lr=1e-3`, `weight_decay=1e-4`, `grad_clip=1.0`,
  `n_epochs=50`, `patience=12` (early stop on val loss). The blend-α direct-
  parametrised params (§3.5) sit in a separate optimiser group with
  `weight_decay=0` and are clamped to $[\varepsilon, 1-\varepsilon]$ with
  $\varepsilon=10^{-3}$ after each step. The ctx-α predictors (§3.5) share
  that group.
- **`alpha_lr_mult=30.0`** (2026-09-01): the blend-α group trains at
  `alpha_lr_mult × lr`. At the default 1.0 α stays pinned to its init (§3.5);
  at 30 both the scalar and ctx-α paths reach their own optima, and the ctx
  predictors develop real weights ($\sum|w|$ 0.19 → 1.46 for `moon`, 2.34 for
  `continuum`) instead of sitting flat. This single change produced most of
  the 2026-09-01 gains.
- **Batching**: `batch_size=512`. All training data (~O(10 MB) after RobustScaler
  standardisation of scores and ctx, fit on train rows only) is staged once
  onto the training device and iterated with `torch.randperm` — no per-batch
  CPU→device transfers.
- **Split**: rows are grouped by `night_id = floor(mjd - 0.5)` and stratified
  by lunar phase into train / val / test partitions via
  `split_indices_by_moon_phase` (defined in `mlp_predictor.ml_utils`). The
  same random seed splits the corpus identically across ensemble members so
  they all see the same held-out test rows.
- **Ensembling**: 10 seeds `(42, 43, ..., 51)` train independently. At
  inference the ensemble mean of the per-seed physical-space predictions is
  returned; the per-row std is also reported as an epistemic-uncertainty
  flag by the `predictive_uncertainty_ensemble` diagnostic.

## 3.8 Diagnostics

The `mlp_predictor.diagnostics.Diagnostics` class runs the diagnostic cells
against a captured globals dict, so the cell bodies (stored verbatim in
`skysub/mlp_predictor/diagnostics_cells/*.py`) stay readable and re-runnable
from the notebook. The active suite covers:

- **Coefficient-space quality**: `coef_residual_vs_value`,
  `naive_baseline` (vs `copy_near` / `near_geo` / `mean_geo`),
  `rmse_dual_diagnostic`, `rmse_worst_stability`.
- **Full-spectrum reconstruction**: `full_spectrum_single_row`,
  `full_spectrum_batch_rmse` (100-row sample, per-row per-component residuals
  with stable per-row colours across all five spectrum panels),
  `worst_recon`, `pipeline_state_check`, `physical_space_cap`.
- **Regime & context slicing**: `per_lunation_drift`, `per_context_slice`,
  `sky_arm_zodi_bias` (verdict keys off the **ML** bias, gated on both a
  slice-relative and an all-test-relative tolerance; a large arm bias with a
  near-zero ML bias is reported as the ML *working*, not as a tuning target).
- **Sigma calibration & ensemble spread**: `resid_over_sigma_per_group`,
  `resid_vs_sigma_per_decile`, `truth_conditioned_sigma_calibration`
  (Mahalanobis $|z|_{\rm joint}$ on the persisted COEF_COV blocks),
  `moon_sigma_investigation` (joint vs marginal SNR by moon regime),
  `predictive_uncertainty_ensemble`, `ensemble_spread_calibration`
  (reliability curve for ensemble std vs actual error).
- **Missing-feature and noise-floor tests**: `residual_ctx_attribution`
  (linear + RF fit of per-row residual RMS on ctx, 5-fold CV R²; also appends
  diagnostic-only `helio_lon_*` probe columns the model never sees, so a
  candidate feature can be tested before spending a training run),
  `sky_arm_disagreement_floor` (ML error vs sky-arm intrinsic disagreement Δ),
  `wavelength_residual_atlas` (aggregated pred − true λ across a 200-row
  sample, absolute + fractional per band; plus a per-coefficient-group
  attribution of the trimmed-mean residual with each group's `flux_share_%`
  and own-units `self_bias_%`, a chi2 gate that drops failed decompositions
  from the sample, and a tail-concentration report).

  **Reading `sky_arm_disagreement_floor` correctly.** The cell's printed
  interpretation says err/Δ ≈ 1 is the floor. That is too optimistic. With
  Δ = ½·RMS(near − far) and both arms carrying independent noise σ,
  Δ ≈ 0.707σ, while a *perfect* predictor still scores err = σ against a sci
  truth carrying its own independent σ — i.e. **err/Δ ≈ 1.41 is the floor**
  for a group whose truth is as noisy as the arms. Reading mesospheric's 1.40
  as headroom is what motivated the failed `mesospheric_group_weight=1.75`
  A/B on 2026-09-01. Values well below 1 (ionospheric ≈ 0.33, zodi ≈ 0.67)
  mean the head is using ctx the arms do not carry.
- **Optional deeper cells**: `swrmse_coef_map`, `wrmse_vs_ctx_correlation`,
  `wrmse_vs_ctx_scatter`, `per_seed_vs_ensemble`. These are commented out by
  default in the notebook (they add ~1 min each) but are supported by the
  `Diagnostics` class the same way.

Each call cell in the notebook opens with two comment lines — `# Targets:`
(what the plot / table shows) and `# Look for:` (what interpretation flags a
regression) — so the diagnostic body can be inspected and rerun in isolation.

## Changelog (split-zodi variant)

> **Numbers before 2026-09-04 are not comparable to numbers after it.** The
> decomposition corpus was regenerated with the moon/zodi identifiability
> constraints (§1.4.3), which moved both the split *and* the moon+zodi sum.
> In particular the near–far moon disagreement fell 12× ($p_{50}$ 0.0554 →
> 0.0045), so any `err/Δ` ratio from the sky-arm floor diagnostic has a
> different denominator and cannot be compared across the boundary. Use the
> `naive_baseline` per-group gains for cross-corpus comparison. Entries for
> the earlier `new-oh` and `moon_zodi_spline` corpora were removed on
> 2026-09-04; `git log` holds them.

- **2026-09-04** — Baseline checkpoint on the `new-oh-2` corpus, plus a
  simplification pass.

  **Corpus.** 17 214 rows → 10 249 after filtering (§1.11), including the new
  moon/zodi reversal gate (−434 rows, 2.5%). Decomposed with the four
  identifiability constraints now default for
  `--fit-model lsf-surface-iterative-split-zodi`.

  **Deployed metrics**, 10-seed ensemble, night-held-out split (n_test = 1113):

  | | value |
  |---|---|
  | ensemble `mean_eRMSE` / `mean_eWRMSE` | 8.273 / 6.041 |
  | seed-to-seed std / ensemble stderr | 0.276 / 0.087 |
  | `naive_baseline` gain vs best naive | moon +23.0%, zodi +41.0%, continuum +10.6%, mesospheric +3.9%, ionospheric +5.6%, atomic +20.1% |
  | group-equal aggregate | +5.5% (ML 8.053 vs `B1_near_geo` 8.518) |
  | `sky_arm_zodi_bias` | 0 of 9 slices carry an ML-specific bias |
  | `wavelength_residual_atlas` RMS\|frac\| | blue 0.59%, mid 1.11%, NIR 0.90% |
  | `sky_arm_disagreement_floor` p50 err/Δ | moon 1.55, zodi 0.35, continuum 1.36, mesospheric 1.39, ionospheric 0.25, atomic 0.85 |

  ML now beats every naive baseline in the all-regime table for the first
  time. The zodi group sits far inside its irreducible band (err/Δ = 0.35
  against an oracle floor of ~1.41).

  **Ablations** (10 seeds each, identical preprocessing and split; gains
  quoted over `B0_copy_near`):

  | change | verdict | evidence |
  |---|---|---|
  | drop the moon-zodi coupling | **rejected** | moon +24.0% → +19.5%, zodi flat. The coupling is the moon-context fix it claims to be, not compensation for the old reversals — its projectors load 3.7× more into moon than zodi, and the damage is moon-only. |
  | `moon_group_weight` 3.0 → 1.5 | neutral | moon −0.3 pp. A scalar group weight is not the lever for the now heavy-tailed moon target (median `moon_tot` 146 vs p99 1.2e5). |
  | drop the flux-space loss | **rejected** | moon −18.5 pp, zodi −18.1 pp, moon-up zodi went negative (−11.2%, worse than copying the near arm), seed std tripled. See §3.6.1 for the per-group split. |
  | `zodi_head_extra_dims` (32,) → () | **adopted** | every group within ±1 pp, aggregate inside seed noise. 7 461 parameters bought nothing measurable. |

  **Removed** (verified bit-identical on a 2-seed regression: `mean_eRMSE`
  8.061867695587885 before and after, and every per-group sRMSE to machine
  precision):

  - the bright-moon-close row-weight boost and its three knobs — carried at
    1.0 (off); row weights are now uniform by construction (§3.6.5);
  - 14 config keys the trainer never read, left behind by an earlier trim.
    Two were active hazards: `moon_zodi_mode` looked like the coupling's
    off-switch and was not, and `alpha_ctx_groups` looked like it restricted
    context-dependent α but the model hardcodes `_ALPHA_CTX_GROUPS`. An A/B
    driven through either would have measured nothing.
    `mlp_predictor.ablations.apply()` now refuses any override the trainer
    does not consume;
  - module docstrings that were removal logs rather than descriptions.

  **Known weaknesses.** The moon target is now bimodal — over half the corpus
  sits at the 0.02 moon-share prior floor, with all moon signal in a minority
  of rows; a per-row relative loss is the natural instrument but the flux-space
  term already supplies that normalisation, so it is not obviously needed. The
  bright-moon decomposition residual (≈1% of continuum, blue-weighted, ∝ FLI)
  is unexplained and is *not* instrumental: its shape does not collapse across
  geometry (fractional spread 0.84, median pairwise shape correlation 0.38).
  Two larger defects sit outside this pipeline: telluric O₂ (A-band −6.5% of
  continuum, B-band +3.1%) and a −4.7% step at the r|z arm boundary.

- **2026-09-03** — Moon/zodi identifiability constraints implemented and made
  the default for the split-zodi decomposition (§1.4.3). Before them the fit
  put the two colours in the wrong order on 164 of 168 moon-up spectra, gave
  the moon family 45% of the continuum with the moon 37° below the horizon, and
  produced a "zodi" tracking lunar illumination at ρ = +0.95 while retaining
  only ρ = 0.09 of its Leinert B₅₀₀ dependence. Full design, math, measured
  effect and rejected alternatives in §1.4.3; standalone write-up in
  `moon_zodi_split_priors_2026-09-03.md`.

- **Architecture provenance.** The isolated zodi and continuum branches, the
  context-dependent blend α, the additive moon-zodi coupling and the
  flux-space loss were each introduced with their own A/B on superseded
  corpora. Their present rationale and measured value live with the
  components themselves (§3.4.1, §3.4.2, §3.4.3, §3.5, §3.6.1) rather than in
  dated entries here; the 2026-09-04 ablation table above is the current
  evidence for keeping each one.


In [ ]:
# --- Package imports ---
%load_ext autoreload
%autoreload 2

import os
os.environ.setdefault("LVMCORE_DIR", "/Users/droryn/prog/lvm/lvmcore")

import numpy as np
import pandas as pd
import plotly.io as pio

# Notebook cwd is skysub/, so mlp_predictor/ and sky_decomp/ are top-level packages.
from mlp_predictor import config, data, wavelengths, compressor, trainer, diagnostics
from mlp_predictor.config import DataConfig, PipelineConfig

# Journal-style plotly axes (as in the pre-refactor notebook).
for _ax in (pio.templates["plotly_white"].layout.xaxis,
            pio.templates["plotly_white"].layout.yaxis):
    _ax.showgrid = False
    _ax.showline = True
    _ax.mirror = True
    _ax.linecolor = "black"
    _ax.linewidth = 1
    _ax.ticks = "inside"
    _ax.zeroline = False
pio.templates.default = "plotly_white"

FACTOR = 1e14

# --- Corpus + artifact locations: the one place to switch corpora -----------
# Every input path -- meta, per-arm coefficients, per-arm decomposition, the
# basis grid and the wavelength cache -- is derived from DECOMP_DATA_ROOT by
# mlp_predictor.config.DataConfig, and the trained ensemble is written beside
# the corpus it was trained on, so a single edit here moves the whole pipeline.
#   'new-oh'   -- decomposed 2026-08-31, BEFORE the moon/zodi shape priors.
#   'new-oh-2' -- re-decomposed with the shape priors (commit 3bd053c).  Both
#                 the moon/zodi split and their sum moved, so this corpus needs
#                 its own wavelength cache, compressors and ensemble; none of
#                 them are transferable from 'new-oh'.
# The wavelength cache is per-root and self-invalidating: a root without
# coef_wavelengths_basis_v4.npz simply rebuilds it on first run.
DECOMP_DATA_ROOT = 'new-oh-2'
ENSEMBLE_FILENAME = 'mlp_ensemble_split_zodi.pt'   # 'new-oh' legacy name: mlp_ensemble_split_zodi_new_oh.pt

cfg = PipelineConfig()
cfg.data.decomp_data_root = DECOMP_DATA_ROOT
SAVED_ENSEMBLE_PATH = f"{DECOMP_DATA_ROOT}/{ENSEMBLE_FILENAME}"

print(f"Corpus:   {cfg.data.decomp_data_root}/{cfg.data.decomp_stem} "
      f"(suffix={cfg.data.decomp_suffix!r})")
print(f"Ensemble: {SAVED_ENSEMBLE_PATH}")

In [ ]:
# --- Load triplet + apply filters + augment context ---
triplet = data.build_triplet_coef_dataset(
    input_fits_path=cfg.data.input_fits_meta,
    sky_near_decomp_fits_path=cfg.data.coef_fits("sky1"),
    sky_far_decomp_fits_path=cfg.data.coef_fits("sky2"),
    sci_decomp_fits_path=cfg.data.coef_fits("sci"),
    context_columns=list(cfg.data.context_columns),
    return_chi2=True,
)
print("Loaded triplet shapes:",
      {k: v.shape for k, v in triplet.items()
       if hasattr(v, "shape") and k in ("coef_near","coef_far","coef_sci","ctx_sci")})

# Native wavelength grid, for the moon/zodi role-reversal check below.
from astropy.io import fits as _fits_w
with _fits_w.open(cfg.data.input_fits_for_basis) as _hw:
    _wave_native = np.asarray(_hw["WAVE"].data, dtype=float)
    _wave_native = _wave_native if _wave_native.ndim == 1 else _wave_native[0]

filtered_triplet = data.apply_triplet_filters(
    triplet,
    thin_every_n=1, chi2_qmax=90.0, chi2_min=0.0, chi2_max=10.0,
    hard_coef_bounds={"feo": (0.0, 10.01), "atom_k": (0.0, 10.01)},
    kappa=6.0, kappa_iter=3, oh_kappa=4.0, oh_kappa_iter=3,
    exclude_field_regions=[data.LMC_EXCLUSION, data.SMC_EXCLUSION],
    # Drop rows where the moon and zodi splines swapped roles in ANY arm: the
    # moon coefficients of such a row describe zodiacal light, so they teach
    # the network the wrong geometry mapping rather than merely adding noise.
    # Tested only where both families carry flux (moon share in [0.05, 0.95]),
    # so dark-time rows -- pinned near a 0.02 moon share by the decomposition
    # priors -- are exempt by construction.
    reversal_decomp_fits={
        "near": cfg.data.decomp_fits("sky1"),
        "far":  cfg.data.decomp_fits("sky2"),
        "sci":  cfg.data.decomp_fits("sci"),
    },
    reversal_wave=_wave_native,
    reversal_min_component_frac=0.05,
    reversal_min_separation=0.0,
)

data._augment_triplet_with_ecliptic(filtered_triplet, force=True, meta_fits_path=cfg.data.input_fits_meta)
data._augment_triplet_with_physics_priors(filtered_triplet, force=True)
print(f"Augmented ctx: n_ctx={len(filtered_triplet['ctx_names'])}")


In [ ]:
# Preview pre- vs post-filter coefficient histograms (no ML state needed).
from mlp_predictor import diagnostics  # local import: survives kernel-restart edge cases

_tmp = diagnostics.Diagnostics(diagnostics.DiagnosticsContext(
    filtered_triplet=filtered_triplet,
    extras={"triplet": triplet},
)).coef_hist_prepost()


In [ ]:
# --- Per-row joint covariance blocks (COEF_COV_MOON / COEF_COV_ZODI) ---
import numpy as _np
from astropy.io import fits as _fits

def _load_cov_for_row_index(decomp_path, row_index):
    row_index = _np.asarray(row_index, dtype=_np.int64)
    with _fits.open(decomp_path) as _h:
        _moon = (_np.asarray(_h["COEF_COV_MOON"].data[row_index], dtype=_np.float64)
                 if "COEF_COV_MOON" in _h else None)
        _zodi = (_np.asarray(_h["COEF_COV_ZODI"].data[row_index], dtype=_np.float64)
                 if "COEF_COV_ZODI" in _h else None)
    return _moon, _zodi

_row_idx_cov = _np.asarray(filtered_triplet["row_index"], dtype=_np.int64)
for _arm, _path in {
    "near": cfg.data.decomp_fits("sky1"),
    "far":  cfg.data.decomp_fits("sky2"),
    "sci":  cfg.data.decomp_fits("sci"),
}.items():
    _m, _z = _load_cov_for_row_index(_path, _row_idx_cov)
    filtered_triplet[f"coef_cov_moon_{_arm}"] = _m
    filtered_triplet[f"coef_cov_zodi_{_arm}"] = _z
print("Per-row joint covariance loaded (or None where COEF_COV_* HDU absent).")


In [ ]:
# --- Populate coefficient wavelengths + effective extinction ---
ext = wavelengths.resolve_wavelengths_and_extinction(
    filtered_triplet,
    cache_path=cfg.data.wavelength_cache_path,
    input_fits_for_basis=cfg.data.input_fits_for_basis,
    use_fitted_extinction=True,
    verbose=True,
)
group_indices = ext.group_indices
N_MOON_KNOTS, SPLIT_ZODI, N_ZODI_KNOTS = wavelengths.infer_spline_knots(
    filtered_triplet["coef_names"])


## Symmetric Dual-Encoder Group-Head Model

Key design choices in this implementation:
1. Shared near/far encoder weights for symmetry and sample efficiency.
2. Fusion vector uses:
   - $e_{mean} = 0.5(e_{near} + e_{far})$
   - $e_{diff} = e_{near} - e_{far}$
   - $|e_{diff}|$
3. Context branch encodes science-context only, now including folded time features and van Rhijn shell factors.
4. Group-specific heads map fused representation into layer-aware coefficient groups rather than a single flat atomic bucket.

**Note.** `DualEncoderGroupHeadMLP` below is the parent architecture. What
the notebook actually trains is `DualEncoderGroupHeadMLPCompressed` (defined
two cells down, together with the compressor fit), which reuses this
architecture verbatim on a per-group compressed coefficient space and emits
signed linear head outputs. See §5.5 in the methods cell above for the
compression pipeline and the empirical argument for switching to it.


In [ ]:
# --- Fit per-group compressors on a moon-phase-stratified split ---
from mlp_predictor.ml_utils import moon_phase_deg_from_ctx, split_indices_by_moon_phase

_moon_phase = moon_phase_deg_from_ctx(filtered_triplet)
_split_tr, _split_va, _split_te = split_indices_by_moon_phase(
    filtered_triplet["obstime_mjd"], _moon_phase, seed=42)

group_compressors, compress_geom_kwargs = compressor.fit_all_group_compressors(
    filtered_triplet, group_indices,
    train_idx=_split_tr, held_idx=_split_va,
    xarm_threshold=compressor.COMPRESSION_XARM_THRESHOLD,
    verbose=True,
)
filtered_triplet["compress_train_idx"] = _split_tr
filtered_triplet["compress_val_idx"] = _split_va
filtered_triplet["compress_test_idx"] = _split_te


In [ ]:
# --- Training config (edit here to override deployed defaults) ---
# Full deployed config expanded in place so every knob is visible; every value
# is initialised to the mlp_predictor.trainer default and can be changed below.

# ----------------------------------------------------------------------
# ----------------------------------------------------------------------
# 2026-08-27c: Phase F -- additive moon-zodi coupling.  Adds a small
# coupling branch (~9k params) that produces a shared latent from the
# moon-zodi ctx union, projected additively into the moon head output
# and the zodi head output via zero-init linear projectors.  Both
# existing paths (moon on shared trunk, zodi on isolated branch) are
# preserved -- this is the "preserve zodi isolation, give moon the
# shared context" fix for Phase E's atlas-mid regression (2026-08-27b).
# At init the projector outputs are zero, so training starts byte-
# identical to Phase A''; the projectors learn to route the coupling.
# To A/B against Phase A'' (no coupling), set
# "moon_zodi_coupling_enabled": False -- or ablations.apply(train_cfg,
# "A1_no_coupling").  The old instruction ("set the four moon_zodi_* entries
# to None") stopped working when the 2026-08-27 trim deleted the no-coupling
# path: it raised on an empty restriction instead of disabling anything.
# Phase E (moon_zodi_mode="shared_branch") is gone from model.py entirely.
# 2026-08-26f: Phase A' landed on top of Phase D.  Adds three explicit
# interaction features to the ctx augment pipeline (computed in
# data._augment_triplet_with_physics_priors):
#   moon_fli_x_phase_cos = moon_fli * moon_phase_cos          # 2nd-order phase
#   moon_sig_x_lon_cos   = moon_signal_proxy * ecl_lon_cos    # moon x zodi geom
#   moon_sig_x_lon_sin   = moon_signal_proxy * ecl_lon_sin    # moon x zodi geom
# All three fed into the isolated continuum branch (see continuum_ctx_restriction
# below).  Attacks the residual_ctx_attribution RF top importances on continuum
# (moon_fli 0.21, moon_phase_cos 0.15, ecl_lon_cos 0.09, moon_signal_proxy 0.05).
# A/B vs 26e (Phase D, same 10 seeds, n_test=1231):
#   mean_eRMSE  21.12 -> 20.98 (-0.7%)  |  mean_eWRMSE 14.98 -> 14.71 (-1.8%)
#   seed std    0.801 -> 0.509 (-36%, most of Phase-D variance recovered)
#   sky-arm continuum p50 err/Delta: all 1.19 -> 1.16, moon_down 1.10 -> 1.08,
#     close_zodi 1.27 -> 1.32 (only regression; still << pre-Phase-D 1.40).
#   wavelength_residual_atlas blue RMS|frac|: 0.535% -> 0.209% (-61%, huge).
#     mid +11% (still 0.63%), NIR +1.5%.  Mean bias grew to -0.4% mid / -0.6% NIR.
#   naive_baseline continuum ML now beats B1_near_geo in every regime (first
#     time): all +2.0%, moon_up +0.2%, moon_down +7.1%, close_zodi +2.4%.
# To A/B against pre-Phase-A' (Phase D only), drop the three trailing entries
# from `continuum_ctx_restriction` below.  The augment always emits them; they
# are simply ignored by the branch when not listed.
# ----------------------------------------------------------------------
# 2026-08-26e: Phase D landed on top of Phase A+B+C.  Widens the isolated
# continuum branch from (64, 32) -> (128, 64) and adds a 64-d hidden layer
# inside the continuum head (+12k targeted params, +2.3% full-model params).
# A/B vs 26d (Phase A+B+C, same 10 seeds, n_test=1231):
#   sky-arm floor, continuum p50 err/Delta:
#     all        1.247 -> 1.186  (-5%)
#     moon_down  1.212 -> 1.095  (-10%, target hit)
#     close_zodi 1.401 -> 1.274  (-9%,  target hit)
#   naive_baseline continuum sRMSE all: 0.01554 -> 0.01461 (-6%, recovers
#     the Phase-B accounting regression; ML now also beats B1_near_geo on
#     continuum moon_down for the first time).
#   wavelength_residual_atlas: blue -12%, NIR -5%, mid +5% (still zero-bias).
#   Aggregate mean_eRMSE 20.99 -> 21.12 (within seed noise); seed std
#     0.41 -> 0.80 (+96%, variance-inflation from the extra 12k params;
#     ensemble stderr still 1.2% of mean).
# To A/B against the pre-Phase-D config (Phase A+B+C only), override:
#     "continuum_branch_dims":     (64, 32)  # current: (128, 64)
#     "continuum_head_extra_dims": ()        # current: (64,)
# To revert further to the pre-Phase-A/B/C baseline, additionally override:
#     "moon_group_weight":         4.0       # current: 2.0
#     "continuum_group_weight":    1.5       # current: 1.0
#     "zodi_head_extra_dims":      ()        # current: (32,)
#     "continuum_ctx_restriction": None      # current: 6-tuple moon-geom
#     "alpha_ctx_features":        None      # current: 3-tuple
#     "alpha_ctx_groups":          None      # current: (moon, zodi, continuum)
# See 2026-08-26d + 2026-08-26e changelog entries for the full A/B tables.
# ----------------------------------------------------------------------
train_cfg = {
    "name": "dual_group_mlp_compressed",

    # Optimisation schedule.
    "n_epochs": 50,
    "batch_size": 512,
    "lr": 1.0e-3,
    "weight_decay": 2.0e-4,
    "patience": 12,

    # Architecture (widths).
    "encoder_dims": (768, 384),
    "ctx_dims": (96,),
    "trunk_dims": (320, 160),
    "head_dim": 192,
    "zodi_head_extra_dims": (),   # A4 (2026-09-04): was (32,).  Ablation showed the 7461-param
                                #  extra layer buys nothing measurable -- every group
                                #  within +/-1pp, aggregate -0.144 (inside the 0.28 seed
                                #  std).  The zodi head sits at err/delta = 0.354, far
                                #  inside its irreducible band, so extra capacity there
                                #  only chases noise.

    # Per-group loss weights m_g (see §3.6.2).
    "moon_group_weight": 3.0,
    "zodi_group_weight": 2.0,
    "continuum_group_weight": 1.0,
    "mesospheric_group_weight": 1.0,
    "ionospheric_group_weight": 1.0,

    # Loss shaping.
    "flux_mse_groups": ("moon", "zodi"),          # §3.6.1 deployed default

    "blend_init_alpha": 0.85,
    "alpha_lr_mult": 30.0,          # enough leverage to get the ctx-dependent alpha to learn efficiently
    
    # Row-weight boosts (see §3.6.5).
    # Phase A'' (2026-08-27): tight-mask 1.5x row-weight boost on bright moon
    # close to sci pointing (fli>=0.90, sep<=30 deg, alt>0).  Boosted subset
    # is only ~90 training rows (vs 500 with the wide 45/0.85 mask), so the
    # training-distribution shift is ~2% -- small enough to avoid the blue-
    # atlas leak the wide masks triggered.  Wide-mask 1.5 and 2.0 A/Bs both
    # showed +45-53% blue RMS|frac| regression regardless of boost magnitude;
    # tight-mask 1.5 keeps it to +23% while gaining -16% on mid RMS|frac|
    # and closing the Phase-A' close_zodi p50 err/Delta regression.
    # A/B vs 26f (Phase A', same 10 seeds, n_test=1231):
    #   sky_arm continuum: all 1.164 -> 1.145, moon_down 1.077 -> 1.054,
    #     close_zodi 1.322 -> 1.284  (all improved -- Phase-A' regression overturned).
    #   atlas mid RMS|frac|: 0.629% -> 0.530% (-16%), mid mean_bias halved -0.38 -> -0.17%.
    #   atlas blue RMS|frac|: 0.209% -> 0.256% (+23%, small; wide masks were +53%).
    #   ensemble mean_eRMSE 20.98 -> 21.11 (+0.6%, within seed noise 0.47).
    # To A/B against the pre-Phase-A'' config (Phase A' only), set:
    #     "bright_moon_close_boost": 1.0,  # current: 1.5

    # COEF_ERR weighting (§3.6.3).
    "coef_err_sigma_floor_rel": dict(trainer.DEFAULT_COEF_ERR_SIGMA_FLOOR_BY_GROUP),

    # 10-seed ensemble.
    "ensemble_seeds": (42, 43, 44, 45, 46, 47, 48, 49, 50, 51),
    
    # Isolated zodi branch context restriction (§3.4.1).
    "zodi_ctx_restriction": (
        "airmass", "vanrhijn_285km",
        "ecl_beta_deg", "ecl_lon_sin", "ecl_lon_cos",
        "zodi_log10_v", "sun_sep", "sun_alt", "alt",
        "moon_alt", "moon_sep",
        "moon_phase_sin", "moon_phase_cos",
        "moon_fli", "moon_up_smooth",
        "moon_airmass_up", "moon_signal_proxy",
    ),
    # Phase B (2026-08-26): moon-regime-conditioned isolated head for continuum.
    # Phase A' (2026-08-26): explicit interaction features added below (from
    # residual_ctx_attribution rankings; computed in data._augment_triplet_with_physics_priors).
    "continuum_ctx_restriction": (
        "moon_alt", "moon_sep",
        "moon_phase_sin", "moon_phase_cos",
        "moon_fli", "airmass",
        "moon_fli_x_phase_cos",
        "moon_sig_x_lon_cos", "moon_sig_x_lon_sin",
    ),
    # Phase D (2026-08-26): widened continuum branch (only group with real
    # headroom above sky-arm floor; residual_ctx_attribution RF R^2 = 0.68).
    # Baseline (64, 32) with () extra layers -> 4k params on the branch.
    # Deployed (128, 64) with (64,) extra layer -> 16k params, +12k targeted.
    "continuum_branch_dims": (128, 64),
    "continuum_head_extra_dims": (64,),
    # Phase A (2026-08-26): context-dependent per-group blend alpha.
    "alpha_ctx_features": (
        "moon_up_smooth", "ecl_beta_deg", "airmass",
    ),
    # Restrict ctx-alpha to the anisotropic groups; others keep scalar alpha.
    # Phase F (2026-08-27c): additive moon-zodi coupling.  Keeps both existing
    # paths intact (moon on shared trunk, zodi on isolated branch, continuum on
    # its own branch).  Adds a small coupling branch (64->32) computing a shared
    # latent from the moon-scatter + zodi-geometry ctx union; the latent is
    # projected additively into the moon head output (Linear(32, n_moon)) and
    # the zodi head output (Linear(32, n_zodi)) via zero-initialised linear
    # projectors, so training starts byte-identical to Phase A''.  This is the
    # "preserve zodi isolation while giving moon the shared context" alternative
    # to Phase E (`moon_zodi_mode="shared_branch"`), which was rejected on
    # 2026-08-27b for the atlas-mid regression it caused.  +9k params.
    # "moon_zodi_ctx_restriction": None, # Phase A baseline
    "moon_zodi_ctx_restriction": (
        "airmass", "vanrhijn_285km",
        "ecl_beta_deg", "ecl_lon_sin", "ecl_lon_cos",
        "zodi_log10_v", "sun_sep",
        "moon_alt", "moon_sep",
        "moon_phase_sin", "moon_phase_cos",
        "moon_fli", "moon_up_smooth",
        "moon_airmass_up", "moon_signal_proxy",
        "moon_fli_x_phase_cos",
        "moon_sig_x_lon_cos", "moon_sig_x_lon_sin",
    ),
    "moon_zodi_coupling_dims": (64, 32),
    # Off-switch for the coupling above, added 2026-09-04 (the no-coupling
    # path had been deleted, so the old "set moon_zodi_* to None" A/B raised).
    # Ablation A1 kept it ON: disabling costs the moon 4.5pp of its gain over
    # B0_copy_near (+24.0% -> +19.5%) while zodi is flat, so the coupling is
    # the moon-context fix its docstring claims, NOT compensation for the
    # moon/zodi reversals the decomposition used to produce.
    # (Phase E's shared_branch knobs lived here; removed with the other dead
    #  keys below -- the code path was deleted on 2026-08-27.)

    # ------------------------------------------------------------------
    # Removed 2026-09-04: 14 config keys the Trainer does not read.  It
    # warned about them on every run ("NOT read by the trainer and will have
    # no effect"); they were left behind by the 2026-08-27 trim, which deleted
    # the code paths but not the config entries.  Two were active hazards for
    # the post-reversal simplification pass: moon_zodi_mode looked like the
    # coupling off-switch and was not (a real one, moon_zodi_coupling_enabled,
    # had to be added), and alpha_ctx_groups looked like it restricted
    # ctx-alpha but the model hardcodes _ALPHA_CTX_GROUPS = ("moon", "zodi",
    # "continuum") -- same value, so behaviour is unchanged.  An A/B driven
    # through any of these would have measured nothing.  The list lives in
    # mlp_predictor.ablations.DEAD_CONFIG_KEYS; ablations.apply() now refuses
    # any override the trainer does not consume.
    #   alpha_ctx_groups, block_cov_loss_groups, flux_mse_eps_frac,
    #   head_extra_dims, high_airmass_boost, moon_down_ecliptic_beta_deg,
    #   moon_down_ecliptic_boost, moon_zodi_branch_dims, moon_zodi_mode,
    #   moon_zodi_moon_head_extra_dims, moon_zodi_zodi_head_extra_dims,
    #   relative_mse_eps_frac, relative_mse_groups, use_coef_err_weights
    # NOTE use_coef_err_weights was hardcoded True and coef_err_sigma_floor_rel
    # (below) is live, so COEF_ERR weighting remains active.
    # ------------------------------------------------------------------
}
print(f"train_cfg: {len(train_cfg)} knobs, {len(train_cfg['ensemble_seeds'])}-seed ensemble, "
      f"flux_mse_groups={train_cfg['flux_mse_groups']}")


In [ ]:
# --- Fit the ensemble, or restore a previously saved one ---------------------
# Gate: when a saved ensemble exists at SAVED_ENSEMBLE_PATH it is deserialised
# and training is skipped entirely; otherwise the ensemble is trained from
# train_cfg above and the next cell writes it to disk.  Set FORCE_RETRAIN=True
# to retrain even when a saved file is present (e.g. after editing train_cfg).
from pathlib import Path as _Path

from mlp_predictor import serialization

# SAVED_ENSEMBLE_PATH is defined in the imports cell next to DECOMP_DATA_ROOT,
# so the model always lands beside the corpus it was trained on.
FORCE_RETRAIN = False

ENSEMBLE_WAS_LOADED = (
    _Path(SAVED_ENSEMBLE_PATH).expanduser().exists() and not FORCE_RETRAIN
)

if ENSEMBLE_WAS_LOADED:
    mlp_artifacts = serialization.load_ensemble(SAVED_ENSEMBLE_PATH)
    _ensemble_members = mlp_artifacts["members"]
    # Training-time only; save_ensemble deliberately does not persist it.
    per_seed_test_metrics = None

    # Refuse a model built against a different corpus or ctx layout -- the
    # network's inputs are positional, so a silent mismatch would produce
    # plausible-looking nonsense rather than an error.
    _saved_coef = list(mlp_artifacts["coef_names"])
    _live_coef = [str(n) for n in filtered_triplet["coef_names"]]
    if _saved_coef != _live_coef:
        raise RuntimeError(
            f"Saved ensemble was trained on a different coefficient basis: "
            f"{len(_saved_coef)} names vs {len(_live_coef)} in filtered_triplet. "
            f"Set FORCE_RETRAIN=True or point SAVED_ENSEMBLE_PATH elsewhere.")
    if list(mlp_artifacts["ctx_names"]) != list(filtered_triplet["ctx_names"]):
        raise RuntimeError(
            "Saved ensemble ctx_names differ from filtered_triplet['ctx_names'] "
            "(context augment changed since training). Set FORCE_RETRAIN=True.")

    # The saved payload omits the train/val/test split by design, but the split
    # is exactly reproducible: Trainer.run_ensemble trains every member on
    # filtered_triplet['compress_*_idx'], which cell 8 derives deterministically
    # from (obstime_mjd, moon_phase, seed=42).
    train_idx = np.asarray(filtered_triplet["compress_train_idx"], dtype=int)
    val_idx = np.asarray(filtered_triplet["compress_val_idx"], dtype=int)
    test_idx = np.asarray(filtered_triplet["compress_test_idx"], dtype=int)
    mlp_artifacts["train_idx"] = train_idx
    mlp_artifacts["val_idx"] = val_idx
    mlp_artifacts["test_idx"] = test_idx

    # Prefer the compressor / geometry / grouping the model was TRAINED with over
    # the freshly fitted ones from cell 8: the network's scores are only
    # meaningful under its own compressor.
    group_compressors = mlp_artifacts["compressors"]
    compress_geom_kwargs = mlp_artifacts["geom_kwargs"]
    group_indices = mlp_artifacts["group_indices"]

    print(f"Restored {len(_ensemble_members)}-member ensemble from "
          f"{SAVED_ENSEMBLE_PATH}; training skipped.")
    print(f"  seeds={list(mlp_artifacts['seeds'])}  "
          f"split: train={train_idx.size} val={val_idx.size} test={test_idx.size}")
    print("  using the saved compressors / geom_kwargs / group_indices "
          "(cell 8's fitted copies are overridden).")
    print("  NB per-seed test metrics and training history are not persisted, so "
          "the training-log")
    print("     tables in this cell are unavailable on the load path.")
else:
    _trainer = trainer.Trainer(cfg=train_cfg)
    artifacts = _trainer.run_ensemble(
        filtered_triplet, group_compressors, group_indices, compress_geom_kwargs,
        input_fits_for_basis=cfg.data.input_fits_for_basis,
        n_moon_knots=N_MOON_KNOTS, split_zodi=SPLIT_ZODI, n_zodi_knots=N_ZODI_KNOTS,
        verbose=True,
    )
    mlp_artifacts = artifacts.mlp_artifacts
    _ensemble_members = artifacts.members
    per_seed_test_metrics = artifacts.per_seed_test_metrics
    train_idx = np.asarray(mlp_artifacts["train_idx"], dtype=int)
    val_idx = np.asarray(mlp_artifacts["val_idx"], dtype=int)
    test_idx = np.asarray(mlp_artifacts["test_idx"], dtype=int)

coef_near_all = np.asarray(filtered_triplet["coef_near"], dtype=np.float32)
coef_far_all  = np.asarray(filtered_triplet["coef_far"],  dtype=np.float32)
coef_sci_all  = np.asarray(filtered_triplet["coef_sci"],  dtype=np.float32)
ctx_near_all  = np.asarray(filtered_triplet["ctx_near"],  dtype=np.float32)
ctx_far_all   = np.asarray(filtered_triplet["ctx_far"],   dtype=np.float32)
ctx_sci_all   = np.asarray(filtered_triplet["ctx_sci"],   dtype=np.float32)
coef_err_near_all = np.asarray(
    filtered_triplet.get("coef_err_near", np.full_like(coef_near_all, np.nan)), dtype=np.float32)
coef_err_far_all = np.asarray(
    filtered_triplet.get("coef_err_far", np.full_like(coef_far_all, np.nan)), dtype=np.float32)
coef_err_sci_all = np.asarray(
    filtered_triplet.get("coef_err_sci", np.full_like(coef_sci_all, np.nan)), dtype=np.float32)

coef_pred_det = trainer.predict_sci_coefficients_default(
    mlp_artifacts,
    coef_near_phys=coef_near_all[test_idx],
    coef_far_phys=coef_far_all[test_idx],
    ctx_near_phys=ctx_near_all[test_idx],
    ctx_far_phys=ctx_far_all[test_idx],
    ctx_sci_phys=ctx_sci_all[test_idx],
).astype(np.float32)


In [ ]:
# --- Persist the trained ensemble to disk for inference ---
# See mlp_predictor.serialization + mlp_predictor.inference for the loader
# and the minimal-input predict API; consumed by
# notebook_example_predict_sky.ipynb.
# SAVED_ENSEMBLE_PATH is defined by the gate in the cell above.  Skipped when
# that gate restored an existing ensemble: there is nothing new to write, and
# re-saving would rewrite the file we just read.
if ENSEMBLE_WAS_LOADED:
    print(f"[save_ensemble] skipped: ensemble was restored from "
          f"{SAVED_ENSEMBLE_PATH} (set FORCE_RETRAIN=True above to retrain).")
else:
    serialization.save_ensemble(mlp_artifacts, SAVED_ENSEMBLE_PATH)


In [ ]:
# --- Build the diagnostics context ---
from mlp_predictor import diagnostics  # local import: survives kernel-restart edge cases

diag_ctx = diagnostics.DiagnosticsContext(
    filtered_triplet=filtered_triplet,
    mlp_artifacts=mlp_artifacts,
    group_compressors=group_compressors,
    group_indices=group_indices,
    geom_kwargs=compress_geom_kwargs,
    ensemble_members=_ensemble_members,
    coef_pred_det=coef_pred_det,
    coef_near_all=coef_near_all,
    coef_far_all=coef_far_all,
    coef_sci_all=coef_sci_all,
    ctx_near_all=ctx_near_all,
    ctx_far_all=ctx_far_all,
    ctx_sci_all=ctx_sci_all,
    coef_err_near_all=coef_err_near_all,
    coef_err_far_all=coef_err_far_all,
    coef_err_sci_all=coef_err_sci_all,
    train_idx=train_idx,
    val_idx=val_idx,
    test_idx=test_idx,
    # Pre-filter triplet + notebook-only bare names the extracted cell bodies read.
    extras={
        "triplet": triplet,
        "coef_wavelengths_a": ext.coef_wavelengths_a,
        "context_cols": list(cfg.data.context_columns),
        "FACTOR": FACTOR,
        "DECOMP_DATA_ROOT": cfg.data.decomp_data_root,
        "DECOMP_STEM": cfg.data.decomp_stem,
        "_DECOMP_SUFFIX": cfg.data.decomp_suffix,
        # Spline-knot counts read by full_spectrum_* cell bodies.
        "N_MOON_KNOTS": N_MOON_KNOTS,
        "SPLIT_ZODI": SPLIT_ZODI,
        "N_ZODI_KNOTS": N_ZODI_KNOTS,
        # Per-seed test metrics DataFrame consumed by naive_baseline.
        # None on the load path: per-seed metrics are not persisted.
        "cmp_df": per_seed_test_metrics,
    },
)
diag = diagnostics.Diagnostics(diag_ctx)


In [ ]:
# Targets: per-coefficient (pred - true) vs true, per group.
# Look for: median off zero => bias; funnel/asymmetry => heteroscedastic residual.
_tmp = diag.coef_residual_vs_value()


## Relationship Visualizations
The following visualization cell is copied from the existing notebook and uses the default predictor interface.

In [ ]:
# diag.relationship_scatter_matrix()


## Full-Spectrum Verification
The following verification cell is copied from the existing notebook and reconstructs a selected science row from predicted coefficients.

In [ ]:
# Targets: reconstruct one every10 row from pred vs true coefs.
# Look for: quality of the fit; residual panels break out moon / zodi / lines / diffuse.
# REQUESTED_ROW = 612
# REQUESTED_ROW = 500
REQUESTED_ROW = 978
# REQUESTED_ROW = 1481
# REQUESTED_ROW = 741 #OK
# REQUESTED_ROW = 742
# REQUESTED_ROW = 830
_tmp = diag.full_spectrum_single_row(row=REQUESTED_ROW, show_moon_zodi_model=True)


In [ ]:
# Targets: 100-row sample: per-row flux-space RMSE (near/far/sci).
# Look for: sci pRMSE distribution; per-component residual per row.
diag._init_globals()
_tmp = diag.full_spectrum_batch_rmse()


In [ ]:
# Targets: the 10 worst reconstructions from the batch above.
# Look for: which regimes (moon-up, bright-moon, high-|beta_ecl|, twilight) dominate the tail.
_tmp = diag.worst_recon()


In [ ]:
# Targets: sanity: predictor + per-group true/pred median and mean bias.
# Look for: any group with mean bias > 5% signals a broken head or lift calibration.
_tmp = diag.pipeline_state_check()


In [ ]:
# Targets: per-seed vs ensemble-mean bias by group.
# Look for: seed-dependent sign flip => under-trained; large seed spread => unstable.
_tmp = diag.per_seed_vs_ensemble()


In [ ]:
# Targets: ML vs copy_near / near_geo / mean_geo baselines, sRMSE per group per regime.
# Look for: ML should beat B2_mean_geo on moon+zodi; look at moon_up / moon_down / close_zodi.
_tmp = diag.naive_baseline()


In [ ]:
# Targets: coefficient-space and pixel-space RMSE side by side.
# Look for: consistency between the two; disagreement points to compressor / geometry issues.
_tmp = diag.rmse_dual_diagnostic()


In [ ]:
# Targets: are the worst rows the same across seeds?.
# Look for: seed-robust worst rows are real failures; seed-varying ones are optimisation noise.
_tmp = diag.rmse_worst_stability()


In [ ]:
# Targets: ML per-lunation mean/max error trend.
# Look for: monotone drift across lunations => solar-activity / seasonal missing feature.
_tmp = diag.per_lunation_drift()


In [ ]:
# Targets: ML error binned by ctx features (moon_alt, |beta_ecl|, airmass, ...).
# Look for: any slice with WRMSE >> global median flags a regime the head under-predicts.
_tmp = diag.per_context_slice()


In [ ]:
# Targets: zodi mean bias per moon-state regime, separately for near and far arms.
# Look for: asymmetric bias => zodi head is being pushed by the wrong arm.
_tmp = diag.sky_arm_zodi_bias()


In [ ]:
# Targets: per-row ensemble std as an epistemic-uncertainty flag.
# Look for: high-sigma rows to inspect qualitatively.
_tmp =diag.predictive_uncertainty_ensemble()


In [ ]:
# Targets: residual / sigma distribution per group (should be ~N(0,1) if calibrated).
# Look for: median |z| ~ 0.7; long tail => under-estimated sigma.
_tmp =diag.resid_over_sigma_per_group()


In [ ]:
# Targets: empirical RMS(err) vs sigma decile (log-log reliability curve).
# Look for: diagonal => calibrated; systematic below => sigma over-estimates error.
_tmp =diag.resid_vs_sigma_per_decile()


In [ ]:
# Targets: Mahalanobis |z|_joint on the persisted COEF_COV_MOON / COEF_COV_ZODI blocks.
# Look for: median |z| ~ 0.7 target; << 0.7 means joint sigma over-estimates.
_tmp =diag.truth_conditioned_sigma_calibration()


In [ ]:
# Targets: joint SNR sqrt(cT Sigma^-1 c) on the moon block per moon regime.
# Look for: compare joint vs marginal SNR; joint is the correct one.
_tmp =diag.moon_sigma_investigation()


In [ ]:
# Targets: verify the runaway upper bound never clips a real prediction.
# Look for: n_clipped should be 0 on train/val/test; max(pred/bound) << 1.
_tmp =diag.physical_space_cap()


In [ ]:
# Producer: builds triplet_full / wrmse_full / is_train / is_valtest /
# is_region_excluded consumed by the next two cells.
# _tmp = diag.swrmse_coef_map()


In [ ]:
# Targets: Pearson + Spearman of full-triplet WRMSE against every ctx feature.
# Look for: strong correlation => head is losing information about that feature.
# _tmp =diag.wrmse_vs_ctx_correlation()


In [ ]:
# Targets: scatter of full-triplet WRMSE vs each ctx feature.
# Look for: regime-dependent WRMSE structure that the correlation numbers hide.
# _tmp =diag.wrmse_vs_ctx_scatter()


In [ ]:
# Targets: linear + RF regression of per-group residual RMS on ctx (5-fold CV R^2).
# Look for: R^2 > 0.10 means the head is under-using ctx features it already sees.
# Missing-feature detector: linear + RF regression on per-group residual RMSE.
_tmp = diag.residual_ctx_attribution()


In [ ]:
# Targets: ML error vs sky-arm intrinsic disagreement (irreducible-noise floor).
# Look for: p50 err/Delta ~ 1 => at the floor; > 2 => headroom; < 1 => ML earns its keep.
# Irreducible noise floor: ML error vs sky-arm intrinsic disagreement.
_tmp = diag.sky_arm_disagreement_floor()


In [ ]:
# Targets: aggregated (pred - true)(lambda) across 200 rows, absolute + fractional.
# Look for: coherent bumps flag specific bands; fractional bias per band signals color miscalibration.
# Physics-space failure map: aggregated pred - true(lambda) across 200 rows.
_tmp = diag.wavelength_residual_atlas()


In [ ]:
# Targets: ensemble std vs actual |pred - true| per group + reliability curve.
# Look for: RMS ratio ~ 1 + Kendall tau > 0.3 => trustworthy sigma; ratio >> 1 => under-diverse ensemble.
# Uncertainty trust: ensemble std vs actual |pred - true| per group.
_tmp = diag.ensemble_spread_calibration()
